In [ ]:
# @title Default title text
# =================================================================================================
# MILESTONE 20.6.3.2
# STABLE-MECHANISM, STRUCTURAL-ORACLE, EVENT-BALANCED TEMPORAL BENCHMARK
# =================================================================================================

from __future__ import annotations

import gc
import json
import math
import random
import warnings

from dataclasses import asdict, dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch

from scipy.stats import spearmanr

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    HistGradientBoostingClassifier,
    IsolationForest,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from transformer_lens import HookedTransformer


warnings.filterwarnings("ignore")


# =================================================================================================
# 1. CONFIGURATION
# =================================================================================================

@dataclass
class Config:
    milestone: str = "M20.6.3.2"

    title: str = (
        "Stable-Mechanism, Structural-Oracle, "
        "Event-Balanced Temporal Benchmark"
    )

    model_name: str = "Qwen/Qwen2.5-0.5B"

    seed: int = 206322

    history_length: int = 8

    forecast_horizons: Tuple[int, ...] = (
        1,
        2,
        3,
        6,
    )

    primary_horizon: int = 3

    stream_length: int = 112

    burn_in: int = 48

    train_streams: int = 40

    calibration_selection_streams: int = 14

    calibration_probability_streams: int = 14

    calibration_policy_streams: int = 14

    validation_streams: int = 18

    test_streams_per_regime: int = 16

    latent_event_threshold: float = 0.72

    observation_noise: float = 0.025

    process_noise: float = 0.018

    activation_batch_size: int = 32

    activation_pca_dimension: int = 12

    oracle_rollouts_preflight: int = 192

    oracle_rollouts_evaluation: int = 256

    order_shuffle_trials: int = 100

    bootstrap_iterations: int = 300

    maximum_policy_fpr: float = 0.08

    minimum_event_recall: float = 0.50

    maximum_false_alert_episodes_per_100: float = 8.0

    minimum_preflight_point_rho: float = 0.40

    minimum_preflight_point_r2: float = 0.10

    minimum_preflight_hazard_auc: float = 0.70

    minimum_point_rho: float = 0.20

    minimum_point_macro_rho: float = 0.12

    minimum_point_r2: float = 0.02

    minimum_hazard_auc: float = 0.65

    minimum_hazard_ap_gain: float = 0.03

    minimum_order_advantage: float = 0.02

    minimum_observability_rho: float = 0.55

    minimum_observability_r2: float = 0.25

    minimum_novelty_auc: float = 0.75

    minimum_novelty_tpr: float = 0.55

    maximum_novelty_fpr: float = 0.15

    output_file: str = (
        "m20_6_3_2_stable_mechanism_structural_oracle.json"
    )


CFG = Config()


KNOWN_FAMILIES = (
    "AR2",
    "DELAYED",
    "OSCILLATORY",
    "MEAN_REVERTING",
)

UNKNOWN_FAMILIES = (
    "ABRUPT_SWITCH",
    "NONLINEAR_NOVEL",
)


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

TORCH_DTYPE = (
    torch.float16
    if DEVICE == "cuda"
    else torch.float32
)


# =================================================================================================
# 2. REPRODUCIBILITY AND UTILITIES
# =================================================================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(
            False
        )
    except Exception:
        pass


seed_everything(CFG.seed)


def print_header(title: str) -> None:
    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)


def to_builtin(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            str(key): to_builtin(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_builtin(item)
            for item in value
        ]

    if isinstance(value, bool):
        return bool(value)

    if isinstance(value, float):
        if not np.isfinite(value):
            return None

        return float(value)

    return value


def safe_spearman(
    y_true: np.ndarray,
    y_prediction: np.ndarray,
) -> float:
    y_true = np.asarray(y_true)
    y_prediction = np.asarray(y_prediction)

    if (
        len(y_true) < 3
        or np.std(y_true) < 1e-12
        or np.std(y_prediction) < 1e-12
    ):
        return 0.0

    value = spearmanr(
        y_true,
        y_prediction,
    ).statistic

    if not np.isfinite(value):
        return 0.0

    return float(value)


def safe_auroc(
    labels: np.ndarray,
    scores: np.ndarray,
) -> Optional[float]:
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    if len(np.unique(labels)) < 2:
        return None

    return float(
        roc_auc_score(
            labels,
            scores,
        )
    )


def safe_auprc(
    labels: np.ndarray,
    scores: np.ndarray,
) -> Optional[float]:
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    if len(np.unique(labels)) < 2:
        return None

    return float(
        average_precision_score(
            labels,
            scores,
        )
    )


def macro_stream_rho(
    y_true: np.ndarray,
    y_prediction: np.ndarray,
    stream_ids: np.ndarray,
) -> float:
    values = []

    for stream_id in np.unique(
        stream_ids
    ):
        mask = (
            stream_ids
            == stream_id
        )

        if np.sum(mask) < 4:
            continue

        values.append(
            safe_spearman(
                y_true[mask],
                y_prediction[mask],
            )
        )

    if not values:
        return 0.0

    return float(
        np.mean(values)
    )


def evaluate_point(
    y_true: np.ndarray,
    y_prediction: np.ndarray,
    stream_ids: np.ndarray,
) -> Dict:
    return {
        "rho": safe_spearman(
            y_true,
            y_prediction,
        ),
        "macro_stream_rho": (
            macro_stream_rho(
                y_true,
                y_prediction,
                stream_ids,
            )
        ),
        "r2": float(
            r2_score(
                y_true,
                y_prediction,
            )
        ),
        "mae": float(
            mean_absolute_error(
                y_true,
                y_prediction,
            )
        ),
    }


def evaluate_hazard(
    labels: np.ndarray,
    probability: np.ndarray,
    threshold: float,
) -> Dict:
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    probability = np.clip(
        np.asarray(
            probability,
            dtype=np.float64,
        ),
        1e-6,
        1.0 - 1e-6,
    )

    prediction = (
        probability
        >= threshold
    ).astype(np.int64)

    tp = int(
        np.sum(
            (prediction == 1)
            & (labels == 1)
        )
    )

    fp = int(
        np.sum(
            (prediction == 1)
            & (labels == 0)
        )
    )

    tn = int(
        np.sum(
            (prediction == 0)
            & (labels == 0)
        )
    )

    fn = int(
        np.sum(
            (prediction == 0)
            & (labels == 1)
        )
    )

    recall = (
        tp / max(tp + fn, 1)
    )

    precision = (
        tp / max(tp + fp, 1)
    )

    fpr = (
        fp / max(fp + tn, 1)
    )

    return {
        "auroc": safe_auroc(
            labels,
            probability,
        ),
        "auprc": safe_auprc(
            labels,
            probability,
        ),
        "prevalence": float(
            np.mean(labels)
        ),
        "recall": float(recall),
        "precision": float(precision),
        "fpr": float(fpr),
        "brier": float(
            brier_score_loss(
                labels,
                probability,
            )
        ),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    }


def sigmoid(
    value: np.ndarray,
) -> np.ndarray:
    value = np.clip(
        value,
        -30.0,
        30.0,
    )

    return (
        1.0
        / (
            1.0
            + np.exp(-value)
        )
    )


# =================================================================================================
# 3. STABLE STREAM-LEVEL MECHANISMS
# =================================================================================================

def sample_stream_parameters(
    family: str,
    rng: np.random.Generator,
) -> Dict[str, float]:
    common = {
        "forcing_amplitude": float(
            rng.uniform(
                0.055,
                0.105,
            )
        ),
        "forcing_frequency": float(
            rng.uniform(
                0.105,
                0.205,
            )
        ),
        "forcing_phase": float(
            rng.uniform(
                -np.pi,
                np.pi,
            )
        ),
        "bias": float(
            rng.uniform(
                -0.012,
                0.012,
            )
        ),
        "scale": 1.0,
    }

    if family == "AR2":
        common.update({
            "a1": float(
                rng.uniform(
                    0.78,
                    0.94,
                )
            ),
            "a2": float(
                rng.uniform(
                    -0.24,
                    -0.08,
                )
            ),
        })

    elif family == "DELAYED":
        common.update({
            "a1": float(
                rng.uniform(
                    0.52,
                    0.68,
                )
            ),
            "delay_weight": float(
                rng.uniform(
                    0.20,
                    0.34,
                )
            ),
            "delay": int(
                rng.choice([2, 3])
            ),
        })

    elif family == "OSCILLATORY":
        common.update({
            "a1": float(
                rng.uniform(
                    1.20,
                    1.38,
                )
            ),
            "a2": float(
                rng.uniform(
                    -0.82,
                    -0.68,
                )
            ),
        })

    elif family == "MEAN_REVERTING":
        common.update({
            "kappa": float(
                rng.uniform(
                    0.18,
                    0.34,
                )
            ),
            "mean": float(
                rng.uniform(
                    -0.10,
                    0.10,
                )
            ),
            "momentum": float(
                rng.uniform(
                    0.04,
                    0.14,
                )
            ),
        })

    elif family == "ABRUPT_SWITCH":
        common.update({
            "pre_a1": float(
                rng.uniform(
                    0.78,
                    0.92,
                )
            ),
            "pre_a2": float(
                rng.uniform(
                    -0.22,
                    -0.10,
                )
            ),
            "post_a1": float(
                rng.uniform(
                    1.10,
                    1.28,
                )
            ),
            "post_a2": float(
                rng.uniform(
                    -0.76,
                    -0.58,
                )
            ),
            "switch_time": int(
                rng.integers(
                    CFG.burn_in + 30,
                    CFG.burn_in + 65,
                )
            ),
        })

    elif family == "NONLINEAR_NOVEL":
        common.update({
            "linear": float(
                rng.uniform(
                    0.52,
                    0.68,
                )
            ),
            "nonlinear": float(
                rng.uniform(
                    0.24,
                    0.38,
                )
            ),
            "delay_weight": float(
                rng.uniform(
                    -0.18,
                    -0.08,
                )
            ),
        })

    else:
        raise ValueError(
            f"Unknown family: {family}"
        )

    return common


def forcing_term(
    time_index: int,
    parameters: Dict[str, float],
) -> float:
    return float(
        parameters["forcing_amplitude"]
        * np.sin(
            parameters["forcing_frequency"]
            * time_index
            + parameters["forcing_phase"]
        )
        + parameters["bias"]
    )


def base_dynamic_mean(
    family: str,
    history: np.ndarray,
    time_index: int,
    parameters: Dict[str, float],
) -> np.ndarray:
    history = np.asarray(
        history,
        dtype=np.float64,
    )

    current = history[..., -1]

    previous = (
        history[..., -2]
        if history.shape[-1] >= 2
        else current
    )

    drive = forcing_term(
        time_index,
        parameters,
    )

    if family == "AR2":
        value = (
            parameters["a1"]
            * current
            + parameters["a2"]
            * previous
            + drive
        )

    elif family == "DELAYED":
        delay = int(
            parameters["delay"]
        )

        delayed = (
            history[..., -1 - delay]
            if history.shape[-1] > delay
            else history[..., 0]
        )

        value = (
            parameters["a1"]
            * current
            + parameters["delay_weight"]
            * delayed
            + drive
        )

    elif family == "OSCILLATORY":
        value = (
            parameters["a1"]
            * current
            + parameters["a2"]
            * previous
            + drive
        )

    elif family == "MEAN_REVERTING":
        value = (
            current
            + parameters["kappa"]
            * (
                parameters["mean"]
                - current
            )
            + parameters["momentum"]
            * (
                current
                - previous
            )
            + drive
        )

    elif family == "ABRUPT_SWITCH":
        if (
            time_index
            < parameters["switch_time"]
        ):
            a1 = parameters["pre_a1"]
            a2 = parameters["pre_a2"]
        else:
            a1 = parameters["post_a1"]
            a2 = parameters["post_a2"]

        value = (
            a1 * current
            + a2 * previous
            + drive
        )

    elif family == "NONLINEAR_NOVEL":
        delayed = (
            history[..., -3]
            if history.shape[-1] >= 3
            else history[..., 0]
        )

        value = (
            parameters["linear"]
            * current
            + parameters["nonlinear"]
            * np.tanh(
                2.4 * current
            )
            + parameters["delay_weight"]
            * delayed
            + drive
        )

    else:
        raise ValueError(family)

    return np.clip(
        value,
        -2.5,
        2.5,
    )


def scaled_dynamic_step(
    family: str,
    scaled_history: np.ndarray,
    time_index: int,
    parameters: Dict[str, float],
    rng: np.random.Generator,
    process_noise: float,
) -> np.ndarray:
    scale = float(
        parameters["scale"]
    )

    base_history = (
        np.asarray(
            scaled_history,
            dtype=np.float64,
        )
        / max(scale, 1e-8)
    )

    base_mean = base_dynamic_mean(
        family,
        base_history,
        time_index,
        parameters,
    )

    noise = rng.normal(
        0.0,
        process_noise,
        size=np.shape(base_mean),
    )

    return (
        scale
        * (
            base_mean
            + noise
        )
    )


# =================================================================================================
# 4. STREAM GENERATION
# =================================================================================================

@dataclass
class StreamRecord:
    stream_id: int

    group_name: str

    family: str

    novel: bool

    parameters: Dict[str, float]

    latent: np.ndarray

    raw: np.ndarray

    activation: Optional[np.ndarray] = None


def generate_base_latent(
    family: str,
    parameters: Dict[str, float],
    total_length: int,
    rng: np.random.Generator,
) -> np.ndarray:
    latent = np.zeros(
        total_length,
        dtype=np.float64,
    )

    latent[:4] = rng.normal(
        0.0,
        0.20,
        size=4,
    )

    for time_index in range(
        4,
        total_length,
    ):
        mean_value = base_dynamic_mean(
            family,
            latent[
                max(
                    0,
                    time_index - 8,
                ):
                time_index
            ],
            time_index,
            parameters,
        )

        latent[time_index] = (
            mean_value
            + rng.normal(
                0.0,
                CFG.process_noise,
            )
        )

    return latent


def amplitude_match_latent(
    latent: np.ndarray,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, float]:
    active = latent[
        CFG.burn_in:
    ]

    reference = float(
        np.quantile(
            np.abs(active),
            0.88,
        )
    )

    target = float(
        rng.uniform(
            0.76,
            0.90,
        )
    )

    scale = (
        target
        / max(reference, 0.08)
    )

    scale = float(
        np.clip(
            scale,
            0.35,
            5.0,
        )
    )

    return (
        latent * scale,
        scale,
    )


def generate_raw_feedback(
    latent: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    total_length = len(latent)

    raw = np.zeros(
        (
            total_length,
            6,
        ),
        dtype=np.float32,
    )

    previous_prediction = 0.0

    for time_index in range(
        total_length
    ):
        previous_latent = (
            latent[time_index - 1]
            if time_index > 0
            else 0.0
        )

        trend = (
            latent[time_index - 1]
            - latent[time_index - 2]
            if time_index > 1
            else 0.0
        )

        prediction = (
            0.68 * previous_latent
            + 0.12 * trend
            + rng.normal(
                0.0,
                CFG.observation_noise,
            )
        )

        sensitivity = float(
            rng.uniform(
                0.74,
                0.90,
            )
        )

        signed_error = (
            sensitivity
            * latent[time_index]
            + rng.normal(
                0.0,
                CFG.observation_noise,
            )
        )

        outcome = (
            prediction
            + signed_error
        )

        confidence = float(
            np.clip(
                0.90
                - 0.34
                * abs(signed_error)
                + rng.normal(
                    0.0,
                    0.025,
                ),
                0.02,
                0.98,
            )
        )

        prediction_change = (
            prediction
            - previous_prediction
        )

        raw[time_index] = np.asarray([
            prediction,
            outcome,
            confidence,
            prediction_change,
            signed_error,
            abs(signed_error),
        ], dtype=np.float32)

        previous_prediction = prediction

    return raw


def generate_group(
    group_name: str,
    number_of_streams: int,
    families: Sequence[str],
    seed: int,
) -> List[StreamRecord]:
    rng = np.random.default_rng(seed)

    total_length = (
        CFG.burn_in
        + CFG.stream_length
        + max(
            CFG.forecast_horizons
        )
        + 2
    )

    streams = []

    for stream_index in range(
        number_of_streams
    ):
        family = families[
            stream_index
            % len(families)
        ]

        parameters = (
            sample_stream_parameters(
                family,
                rng,
            )
        )

        base_latent = (
            generate_base_latent(
                family,
                parameters,
                total_length,
                rng,
            )
        )

        latent, scale = (
            amplitude_match_latent(
                base_latent,
                rng,
            )
        )

        parameters["scale"] = scale

        raw = generate_raw_feedback(
            latent,
            rng,
        )

        streams.append(
            StreamRecord(
                stream_id=stream_index,
                group_name=group_name,
                family=family,
                novel=(
                    family
                    in UNKNOWN_FAMILIES
                ),
                parameters=parameters,
                latent=latent.astype(
                    np.float32
                ),
                raw=raw,
            )
        )

    return streams


# =================================================================================================
# 5. INDEPENDENT GROUPS
# =================================================================================================

GROUPS: Dict[str, List[StreamRecord]] = {
    "TRAIN_MIXED_KNOWN": generate_group(
        "TRAIN_MIXED_KNOWN",
        CFG.train_streams,
        KNOWN_FAMILIES,
        CFG.seed + 1_000,
    ),
    "CALIBRATION_SELECTION": generate_group(
        "CALIBRATION_SELECTION",
        CFG.calibration_selection_streams,
        KNOWN_FAMILIES,
        CFG.seed + 2_000,
    ),
    "CALIBRATION_PROBABILITY": generate_group(
        "CALIBRATION_PROBABILITY",
        CFG.calibration_probability_streams,
        KNOWN_FAMILIES,
        CFG.seed + 3_000,
    ),
    "CALIBRATION_POLICY": generate_group(
        "CALIBRATION_POLICY",
        CFG.calibration_policy_streams,
        KNOWN_FAMILIES,
        CFG.seed + 4_000,
    ),
    "VALIDATION": generate_group(
        "VALIDATION",
        CFG.validation_streams,
        KNOWN_FAMILIES,
        CFG.seed + 5_000,
    ),
    "AR2_TRANSFER": generate_group(
        "AR2_TRANSFER",
        CFG.test_streams_per_regime,
        ("AR2",),
        CFG.seed + 6_000,
    ),
    "DELAYED_TRANSFER": generate_group(
        "DELAYED_TRANSFER",
        CFG.test_streams_per_regime,
        ("DELAYED",),
        CFG.seed + 7_000,
    ),
    "OSCILLATORY_TRANSFER": generate_group(
        "OSCILLATORY_TRANSFER",
        CFG.test_streams_per_regime,
        ("OSCILLATORY",),
        CFG.seed + 8_000,
    ),
    "MEAN_REVERTING_TRANSFER": generate_group(
        "MEAN_REVERTING_TRANSFER",
        CFG.test_streams_per_regime,
        ("MEAN_REVERTING",),
        CFG.seed + 9_000,
    ),
    "ABRUPT_UNKNOWN": generate_group(
        "ABRUPT_UNKNOWN",
        CFG.test_streams_per_regime,
        ("ABRUPT_SWITCH",),
        CFG.seed + 10_000,
    ),
    "MECHANISM_NOVEL": generate_group(
        "MECHANISM_NOVEL",
        CFG.test_streams_per_regime,
        ("NONLINEAR_NOVEL",),
        CFG.seed + 11_000,
    ),
}


# =================================================================================================
# 6. TEMPORAL DATASETS
# =================================================================================================

def build_temporal_dataset(
    streams: Sequence[StreamRecord],
    horizon: int,
) -> Dict:
    raw_windows = []
    activation_windows = []

    y_point = []
    y_hazard = []
    hazard_eligible = []

    stream_ids = []
    family_names = []
    novelty_labels = []

    row_times = []
    stream_references = []

    current_latent = []
    current_proxy = []

    threshold = (
        CFG.latent_event_threshold
    )

    for global_stream_id, stream in enumerate(
        streams
    ):
        start = (
            CFG.burn_in
            + CFG.history_length
            - 1
        )

        stop = (
            CFG.burn_in
            + CFG.stream_length
        )

        for time_index in range(
            start,
            stop
        ):
            if (
                time_index + horizon
                >= len(stream.latent)
            ):
                continue

            history_slice = slice(
                time_index
                - CFG.history_length
                + 1,
                time_index + 1,
            )

            future = stream.latent[
                time_index + 1:
                time_index + horizon + 1
            ]

            raw_windows.append(
                stream.raw[
                    history_slice
                ]
            )

            if (
                stream.activation
                is not None
            ):
                activation_windows.append(
                    stream.activation[
                        history_slice
                    ]
                )

            current = float(
                stream.latent[
                    time_index
                ]
            )

            eligible = bool(
                abs(current)
                < threshold
            )

            event = bool(
                eligible
                and np.any(
                    np.abs(future)
                    >= threshold
                )
            )

            y_point.append(
                float(
                    stream.latent[
                        time_index + horizon
                    ]
                )
            )

            y_hazard.append(
                int(event)
            )

            hazard_eligible.append(
                eligible
            )

            stream_ids.append(
                global_stream_id
            )

            family_names.append(
                stream.family
            )

            novelty_labels.append(
                int(stream.novel)
            )

            row_times.append(
                time_index
            )

            stream_references.append(
                stream
            )

            current_latent.append(
                current
            )

            current_proxy.append(
                float(
                    stream.raw[
                        time_index,
                        4,
                    ]
                )
            )

    dataset = {
        "X_raw": np.asarray(
            raw_windows,
            dtype=np.float32,
        ),
        "y_point": np.asarray(
            y_point,
            dtype=np.float32,
        ),
        "y_hazard": np.asarray(
            y_hazard,
            dtype=np.int64,
        ),
        "hazard_eligible": np.asarray(
            hazard_eligible,
            dtype=bool,
        ),
        "stream_ids": np.asarray(
            stream_ids,
            dtype=np.int64,
        ),
        "family_names": np.asarray(
            family_names,
            dtype=object,
        ),
        "novelty_labels": np.asarray(
            novelty_labels,
            dtype=np.int64,
        ),
        "row_times": np.asarray(
            row_times,
            dtype=np.int64,
        ),
        "stream_references": (
            stream_references
        ),
        "current_latent": np.asarray(
            current_latent,
            dtype=np.float32,
        ),
        "current_proxy": np.asarray(
            current_proxy,
            dtype=np.float32,
        ),
        "horizon": int(horizon),
    }

    if activation_windows:
        dataset["X_activation"] = (
            np.asarray(
                activation_windows,
                dtype=np.float32,
            )
        )

    return dataset


RAW_DATASETS_BY_HORIZON = {
    horizon: {
        name: build_temporal_dataset(
            streams,
            horizon,
        )
        for name, streams in GROUPS.items()
    }
    for horizon in CFG.forecast_horizons
}


# =================================================================================================
# 7. STRUCTURAL MONTE CARLO ORACLE
# =================================================================================================

def structural_oracle(
    dataset: Dict,
    rollouts: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)

    number_of_rows = len(
        dataset["y_point"]
    )

    horizon = int(
        dataset["horizon"]
    )

    point_prediction = np.zeros(
        number_of_rows,
        dtype=np.float64,
    )

    hazard_probability = np.zeros(
        number_of_rows,
        dtype=np.float64,
    )

    threshold = (
        CFG.latent_event_threshold
    )

    for row_index in range(
        number_of_rows
    ):
        stream = dataset[
            "stream_references"
        ][row_index]

        time_index = int(
            dataset["row_times"][
                row_index
            ]
        )

        history = stream.latent[
            time_index
            - CFG.history_length
            + 1:
            time_index + 1
        ].astype(np.float64)

        rollout_history = np.repeat(
            history[None, :],
            rollouts,
            axis=0,
        )

        crossed = np.zeros(
            rollouts,
            dtype=bool,
        )

        for lead in range(
            1,
            horizon + 1,
        ):
            next_value = (
                scaled_dynamic_step(
                    stream.family,
                    rollout_history,
                    time_index + lead,
                    stream.parameters,
                    rng,
                    CFG.process_noise,
                )
            )

            crossed |= (
                np.abs(next_value)
                >= threshold
            )

            rollout_history = (
                np.concatenate([
                    rollout_history[:, 1:],
                    next_value[:, None],
                ], axis=1)
            )

        point_prediction[
            row_index
        ] = float(
            np.mean(
                rollout_history[:, -1]
            )
        )

        if dataset[
            "hazard_eligible"
        ][row_index]:
            hazard_probability[
                row_index
            ] = float(
                np.mean(crossed)
            )
        else:
            hazard_probability[
                row_index
            ] = 0.0

    return (
        point_prediction.astype(
            np.float32
        ),
        hazard_probability.astype(
            np.float32
        ),
    )


# =================================================================================================
# 8. RAW-ONLY PREFLIGHT
# =================================================================================================

print_header(
    "🔬 MILESTONE 20.6.3.2 — "
    "STABLE-MECHANISM STRUCTURAL-ORACLE PREFLIGHT"
)

print(f"Device                 : {DEVICE.upper()}")
print(f"Model                  : {CFG.model_name}")
print(f"History length         : {CFG.history_length}")
print(f"Forecast horizons      : {CFG.forecast_horizons}")
print(f"Primary horizon        : {CFG.primary_horizon}")
print(f"Latent threshold       : {CFG.latent_event_threshold}")
print(f"Global seed            : {CFG.seed}")


preflight_results = {}

print_header(
    "🧪 PREFLIGHT A — STRUCTURAL FORECASTABILITY"
)

for horizon in CFG.forecast_horizons:
    dataset = RAW_DATASETS_BY_HORIZON[
        horizon
    ]["VALIDATION"]

    point_oracle_prediction, hazard_oracle_probability = (
        structural_oracle(
            dataset,
            CFG.oracle_rollouts_preflight,
            CFG.seed + 20_000 + horizon,
        )
    )

    point_metrics = evaluate_point(
        dataset["y_point"],
        point_oracle_prediction,
        dataset["stream_ids"],
    )

    mask = dataset[
        "hazard_eligible"
    ]

    hazard_metrics = evaluate_hazard(
        dataset["y_hazard"][mask],
        hazard_oracle_probability[mask],
        threshold=0.5,
    )

    hazard_auc = (
        hazard_metrics["auroc"]
        if hazard_metrics["auroc"]
        is not None
        else 0.0
    )

    passed = bool(
        point_metrics["rho"]
        >= CFG.minimum_preflight_point_rho
        and point_metrics["r2"]
        >= CFG.minimum_preflight_point_r2
        and hazard_auc
        >= CFG.minimum_preflight_hazard_auc
    )

    preflight_results[horizon] = {
        "point": point_metrics,
        "hazard": hazard_metrics,
        "pass": passed,
    }

    print(
        f"H={horizon:<2} | "
        f"Point rho={point_metrics['rho']:.4f} | "
        f"Macro={point_metrics['macro_stream_rho']:.4f} | "
        f"R²={point_metrics['r2']:.4f} | "
        f"Hazard AUC={hazard_auc:.4f} | "
        f"AP={(hazard_metrics['auprc'] or 0.0):.4f} | "
        f"Prev={hazard_metrics['prevalence']:.4f} | "
        f"{'PASS' if passed else 'FAIL'}"
    )


primary_preflight_passed = bool(
    preflight_results[
        CFG.primary_horizon
    ]["pass"]
)

if not primary_preflight_passed:
    failure_telemetry = {
        "milestone": CFG.milestone,
        "title": CFG.title,
        "configuration": asdict(CFG),
        "status": (
            "PREFLIGHT_FAILED_BEFORE_ACTIVATION_EXTRACTION"
        ),
        "preflight": preflight_results,
    }

    with open(
        CFG.output_file,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            to_builtin(
                failure_telemetry
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    raise RuntimeError(
        "Primary structural-oracle preflight failed. "
        "Activation extraction was intentionally skipped."
    )


print(
    "\n✅ Primary benchmark is structurally "
    "forecastable. Activation extraction may proceed."
)


# =================================================================================================
# 9. ACTIVATION EXTRACTION
# =================================================================================================

def feedback_prompt(
    raw_row: np.ndarray,
) -> str:
    return (
        "Feedback record\n"
        f"prediction={raw_row[0]:+08.4f}\n"
        f"outcome={raw_row[1]:+08.4f}\n"
        f"confidence={raw_row[2]:+08.4f}\n"
        f"prediction_change={raw_row[3]:+08.4f}\n"
        f"signed_error={raw_row[4]:+08.4f}\n"
        f"absolute_error={raw_row[5]:+08.4f}\n"
        "Assess the internal feedback state:"
    )


print_header(
    "⏳ LOADING TRANSFORMERLENS MODEL"
)

model = HookedTransformer.from_pretrained(
    CFG.model_name,
    device=DEVICE,
    dtype=TORCH_DTYPE,
)

model.eval()

number_of_layers = int(
    model.cfg.n_layers
)

valid_layers = sorted(
    set(
        np.linspace(
            0,
            number_of_layers - 1,
            8,
            dtype=int,
        ).tolist()
    )
)

try:
    model.tokenizer.padding_side = "left"

    if (
        model.tokenizer.pad_token_id
        is None
    ):
        model.tokenizer.pad_token = (
            model.tokenizer.eos_token
        )
except Exception:
    pass


def extract_stream_activations(
    streams: Sequence[StreamRecord],
    layers: Sequence[int],
) -> Dict[int, List[np.ndarray]]:
    layer_outputs = {
        layer: []
        for layer in layers
    }

    for stream in streams:
        prompts = [
            feedback_prompt(row)
            for row in stream.raw
        ]

        collected = {
            layer: []
            for layer in layers
        }

        for start in range(
            0,
            len(prompts),
            CFG.activation_batch_size,
        ):
            batch_prompts = prompts[
                start:
                start
                + CFG.activation_batch_size
            ]

            tokens = model.to_tokens(
                batch_prompts,
                prepend_bos=True,
            )

            hook_names = {
                f"blocks.{layer}.hook_resid_post"
                for layer in layers
            }

            with torch.inference_mode():
                _, cache = model.run_with_cache(
                    tokens,
                    names_filter=lambda name: (
                        name in hook_names
                    ),
                )

            for layer in layers:
                value = cache[
                    f"blocks.{layer}.hook_resid_post"
                ][:, -1, :]

                collected[layer].append(
                    value.detach()
                    .float()
                    .cpu()
                    .numpy()
                )

            del cache
            del tokens

        for layer in layers:
            layer_outputs[layer].append(
                np.concatenate(
                    collected[layer],
                    axis=0,
                ).astype(np.float32)
            )

    return layer_outputs


print(
    f"✅ Model loaded. "
    f"Valid layers: {valid_layers}"
)

print(
    "\n🔬 Extracting train/selection "
    "activations for layer selection..."
)

train_layer_activations = (
    extract_stream_activations(
        GROUPS["TRAIN_MIXED_KNOWN"],
        valid_layers,
    )
)

selection_layer_activations = (
    extract_stream_activations(
        GROUPS[
            "CALIBRATION_SELECTION"
        ],
        valid_layers,
    )
)


# =================================================================================================
# 10. LAYER SELECTION USING ACTIVATION-ONLY OBSERVABILITY
# =================================================================================================

def activation_rows_and_targets(
    streams: Sequence[StreamRecord],
    activation_list: Sequence[np.ndarray],
) -> Tuple[np.ndarray, np.ndarray]:
    rows = []
    targets = []

    for stream, activation in zip(
        streams,
        activation_list,
    ):
        start = CFG.burn_in

        stop = (
            CFG.burn_in
            + CFG.stream_length
        )

        rows.append(
            activation[
                start:stop
            ]
        )

        targets.append(
            stream.latent[
                start:stop
            ]
        )

    return (
        np.concatenate(
            rows,
            axis=0,
        ),
        np.concatenate(
            targets,
            axis=0,
        ),
    )


layer_observability_results = {}
layer_transformers = {}

print_header(
    "🧪 TEST A — ACTIVATION-ONLY FEEDBACK OBSERVABILITY"
)

for layer in valid_layers:
    train_x, train_y = (
        activation_rows_and_targets(
            GROUPS[
                "TRAIN_MIXED_KNOWN"
            ],
            train_layer_activations[
                layer
            ],
        )
    )

    selection_x, selection_y = (
        activation_rows_and_targets(
            GROUPS[
                "CALIBRATION_SELECTION"
            ],
            selection_layer_activations[
                layer
            ],
        )
    )

    pca_dimension = min(
        CFG.activation_pca_dimension,
        train_x.shape[1],
        train_x.shape[0] - 1,
    )

    pca = PCA(
        n_components=pca_dimension,
        random_state=CFG.seed,
    )

    train_reduced = (
        pca.fit_transform(
            train_x
        )
    )

    selection_reduced = (
        pca.transform(
            selection_x
        )
    )

    decoder = Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "ridge",
            Ridge(alpha=2.0),
        ),
    ])

    decoder.fit(
        train_reduced,
        train_y,
    )

    prediction = decoder.predict(
        selection_reduced
    )

    metrics = {
        "rho": safe_spearman(
            selection_y,
            prediction,
        ),
        "r2": float(
            r2_score(
                selection_y,
                prediction,
            )
        ),
        "mae": float(
            mean_absolute_error(
                selection_y,
                prediction,
            )
        ),
    }

    layer_observability_results[
        layer
    ] = metrics

    layer_transformers[
        layer
    ] = {
        "pca": pca,
        "decoder": decoder,
    }

    print(
        f"Layer {layer:<2} | "
        f"rho={metrics['rho']:.4f} | "
        f"R²={metrics['r2']:.4f} | "
        f"MAE={metrics['mae']:.4f}"
    )


BEST_FEEDBACK_LAYER = max(
    valid_layers,
    key=lambda layer: (
        layer_observability_results[
            layer
        ]["r2"]
    ),
)

activation_pca = layer_transformers[
    BEST_FEEDBACK_LAYER
]["pca"]

print(
    f"\n🏆 Selected activation-only "
    f"feedback layer: {BEST_FEEDBACK_LAYER}"
)


# =================================================================================================
# 11. EXTRACT SELECTED LAYER FOR REMAINING GROUPS
# =================================================================================================

for group_name, streams in GROUPS.items():
    if group_name == "TRAIN_MIXED_KNOWN":
        raw_activation_list = (
            train_layer_activations[
                BEST_FEEDBACK_LAYER
            ]
        )

    elif group_name == "CALIBRATION_SELECTION":
        raw_activation_list = (
            selection_layer_activations[
                BEST_FEEDBACK_LAYER
            ]
        )

    else:
        print(
            f"  Extracting {group_name}..."
        )

        raw_activation_list = (
            extract_stream_activations(
                streams,
                (BEST_FEEDBACK_LAYER,),
            )[BEST_FEEDBACK_LAYER]
        )

    for stream, raw_activation in zip(
        streams,
        raw_activation_list,
    ):
        stream.activation = (
            activation_pca.transform(
                raw_activation
            ).astype(np.float32)
        )


del train_layer_activations
del selection_layer_activations

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# =================================================================================================
# 12. REBUILD DATASETS WITH ACTIVATIONS
# =================================================================================================

DATASETS_BY_HORIZON = {
    horizon: {
        name: build_temporal_dataset(
            streams,
            horizon,
        )
        for name, streams in GROUPS.items()
    }
    for horizon in CFG.forecast_horizons
}

DATASETS = DATASETS_BY_HORIZON[
    CFG.primary_horizon
]

train_dataset = DATASETS[
    "TRAIN_MIXED_KNOWN"
]

selection_dataset = DATASETS[
    "CALIBRATION_SELECTION"
]

probability_dataset = DATASETS[
    "CALIBRATION_PROBABILITY"
]

policy_dataset = DATASETS[
    "CALIBRATION_POLICY"
]

validation_dataset = DATASETS[
    "VALIDATION"
]


# =================================================================================================
# 13. FEATURE VIEWS
# =================================================================================================

def ordered_raw_features(
    dataset: Dict,
) -> np.ndarray:
    return dataset[
        "X_raw"
    ].reshape(
        len(
            dataset["X_raw"]
        ),
        -1,
    )


def latest_raw_features(
    dataset: Dict,
) -> np.ndarray:
    return dataset[
        "X_raw"
    ][:, -1, :]


def bag_raw_features(
    dataset: Dict,
) -> np.ndarray:
    raw = dataset["X_raw"]

    return np.concatenate([
        np.mean(
            raw,
            axis=1,
        ),
        np.std(
            raw,
            axis=1,
        ),
        raw[:, -1, :],
    ], axis=1)


def slope_raw_features(
    dataset: Dict,
) -> np.ndarray:
    raw = dataset["X_raw"]

    time_axis = np.arange(
        raw.shape[1],
        dtype=np.float32,
    )

    centered_time = (
        time_axis
        - np.mean(time_axis)
    )

    denominator = float(
        np.sum(
            centered_time ** 2
        )
    )

    slope = np.sum(
        raw
        * centered_time[
            None,
            :,
            None,
        ],
        axis=1,
    ) / max(
        denominator,
        1e-8,
    )

    return np.concatenate([
        raw[:, -1, :],
        slope,
        raw[:, -1, :]
        - raw[:, 0, :],
    ], axis=1)


def activation_ordered_features(
    dataset: Dict,
) -> np.ndarray:
    activation = dataset[
        "X_activation"
    ]

    return activation.reshape(
        len(activation),
        -1,
    )


def combined_ordered_features(
    dataset: Dict,
) -> np.ndarray:
    return np.concatenate([
        ordered_raw_features(
            dataset
        ),
        activation_ordered_features(
            dataset
        ),
    ], axis=1)


def feature_view(
    dataset: Dict,
    model_name: str,
) -> np.ndarray:
    if model_name == "RAW_LATEST":
        return latest_raw_features(
            dataset
        )

    if model_name == "RAW_BAG":
        return bag_raw_features(
            dataset
        )

    if model_name == "RAW_SLOPE":
        return slope_raw_features(
            dataset
        )

    if model_name == "RAW_ORDERED":
        return ordered_raw_features(
            dataset
        )

    if model_name == "ACTIVATION_ORDERED":
        return activation_ordered_features(
            dataset
        )

    if model_name == "COMBINED_ORDERED":
        return combined_ordered_features(
            dataset
        )

    raise ValueError(
        model_name
    )


# =================================================================================================
# 14. POINT AND HAZARD MODEL SELECTION
# =================================================================================================

MODEL_NAMES = (
    "RAW_LATEST",
    "RAW_BAG",
    "RAW_SLOPE",
    "RAW_ORDERED",
    "ACTIVATION_ORDERED",
    "COMBINED_ORDERED",
)

CAUSAL_MODEL_NAMES = (
    "RAW_LATEST",
    "RAW_BAG",
    "RAW_SLOPE",
    "RAW_ORDERED",
)


def make_point_model(
    model_name: str,
):
    if model_name in (
        "RAW_ORDERED",
        "COMBINED_ORDERED",
        "ACTIVATION_ORDERED",
    ):
        return ExtraTreesRegressor(
            n_estimators=320,
            min_samples_leaf=5,
            max_features=0.75,
            random_state=CFG.seed,
            n_jobs=-1,
        )

    return Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "ridge",
            Ridge(alpha=3.0),
        ),
    ])


def make_hazard_model(
    model_name: str,
):
    if model_name in (
        "RAW_ORDERED",
        "COMBINED_ORDERED",
        "ACTIVATION_ORDERED",
    ):
        return ExtraTreesClassifier(
            n_estimators=360,
            min_samples_leaf=5,
            class_weight="balanced",
            max_features=0.75,
            random_state=CFG.seed,
            n_jobs=-1,
        )

    return Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "logistic",
            LogisticRegression(
                C=0.7,
                class_weight="balanced",
                max_iter=3000,
                random_state=CFG.seed,
            ),
        ),
    ])


def model_score(
    model_instance,
    features: np.ndarray,
) -> np.ndarray:
    if hasattr(
        model_instance,
        "predict_proba",
    ):
        probability = (
            model_instance.predict_proba(
                features
            )[:, 1]
        )

        probability = np.clip(
            probability,
            1e-6,
            1.0 - 1e-6,
        )

        return np.log(
            probability
            / (
                1.0
                - probability
            )
        )

    if hasattr(
        model_instance,
        "decision_function",
    ):
        return np.asarray(
            model_instance.decision_function(
                features
            )
        )

    return np.asarray(
        model_instance.predict(
            features
        )
    )


point_models = {}
hazard_models = {}

point_selection_results = {}
hazard_selection_results = {}

print_header(
    "🧪 TEST B — DISENTANGLED MODEL SELECTION"
)

print("\nPoint models:")

for model_name in MODEL_NAMES:
    point_model = make_point_model(
        model_name
    )

    point_model.fit(
        feature_view(
            train_dataset,
            model_name,
        ),
        train_dataset["y_point"],
    )

    prediction = point_model.predict(
        feature_view(
            selection_dataset,
            model_name,
        )
    )

    metrics = evaluate_point(
        selection_dataset["y_point"],
        prediction,
        selection_dataset["stream_ids"],
    )

    point_models[model_name] = (
        point_model
    )

    point_selection_results[
        model_name
    ] = metrics

    channel = (
        "CAUSAL"
        if model_name
        in CAUSAL_MODEL_NAMES
        else "DIAGNOSTIC"
    )

    print(
        f"  {model_name:<20} "
        f"[{channel:<10}] | "
        f"rho={metrics['rho']:.4f} | "
        f"Macro={metrics['macro_stream_rho']:.4f} | "
        f"R²={metrics['r2']:.4f}"
    )


train_hazard_mask = train_dataset[
    "hazard_eligible"
]

selection_hazard_mask = selection_dataset[
    "hazard_eligible"
]

print("\nHazard models:")

for model_name in MODEL_NAMES:
    hazard_model = make_hazard_model(
        model_name
    )

    hazard_model.fit(
        feature_view(
            train_dataset,
            model_name,
        )[train_hazard_mask],
        train_dataset[
            "y_hazard"
        ][train_hazard_mask],
    )

    score = model_score(
        hazard_model,
        feature_view(
            selection_dataset,
            model_name,
        )[selection_hazard_mask],
    )

    probability = sigmoid(score)

    metrics = evaluate_hazard(
        selection_dataset[
            "y_hazard"
        ][selection_hazard_mask],
        probability,
        threshold=0.5,
    )

    hazard_models[
        model_name
    ] = hazard_model

    hazard_selection_results[
        model_name
    ] = metrics

    channel = (
        "CAUSAL"
        if model_name
        in CAUSAL_MODEL_NAMES
        else "DIAGNOSTIC"
    )

    print(
        f"  {model_name:<20} "
        f"[{channel:<10}] | "
        f"AUC={(metrics['auroc'] or 0.0):.4f} | "
        f"AP={(metrics['auprc'] or 0.0):.4f}"
    )


BEST_POINT_MODEL_NAME = max(
    CAUSAL_MODEL_NAMES,
    key=lambda name: (
        point_selection_results[
            name
        ]["rho"]
        + 0.40
        * point_selection_results[
            name
        ]["macro_stream_rho"]
    ),
)

BEST_HAZARD_MODEL_NAME = max(
    CAUSAL_MODEL_NAMES,
    key=lambda name: (
        hazard_selection_results[
            name
        ]["auroc"]
        or 0.0
    ),
)

selected_point_model = (
    point_models[
        BEST_POINT_MODEL_NAME
    ]
)

selected_hazard_model = (
    hazard_models[
        BEST_HAZARD_MODEL_NAME
    ]
)

print(
    f"\n🏆 Selected causal point model : "
    f"{BEST_POINT_MODEL_NAME}"
)

print(
    f"🏆 Selected causal hazard model: "
    f"{BEST_HAZARD_MODEL_NAME}"
)


# =================================================================================================
# 15. INDEPENDENT CHANNEL CALIBRATION
# =================================================================================================

class PlattCalibrator:
    def __init__(self):
        self.model = LogisticRegression(
            C=1.0,
            max_iter=3000,
            random_state=CFG.seed,
        )

    def fit(
        self,
        score: np.ndarray,
        labels: np.ndarray,
    ) -> "PlattCalibrator":
        self.model.fit(
            np.asarray(score).reshape(
                -1,
                1,
            ),
            labels,
        )

        return self

    def predict(
        self,
        score: np.ndarray,
    ) -> np.ndarray:
        return self.model.predict_proba(
            np.asarray(score).reshape(
                -1,
                1,
            )
        )[:, 1]


probability_mask = probability_dataset[
    "hazard_eligible"
]

channel_calibrators = {}

for model_name in MODEL_NAMES:
    score = model_score(
        hazard_models[model_name],
        feature_view(
            probability_dataset,
            model_name,
        )[probability_mask],
    )

    calibrator = PlattCalibrator()

    calibrator.fit(
        score,
        probability_dataset[
            "y_hazard"
        ][probability_mask],
    )

    channel_calibrators[
        model_name
    ] = calibrator


def calibrated_hazard_probability(
    dataset: Dict,
    model_name: str,
) -> np.ndarray:
    mask = dataset[
        "hazard_eligible"
    ]

    probability = np.zeros(
        len(
            dataset["y_hazard"]
        ),
        dtype=np.float64,
    )

    score = model_score(
        hazard_models[
            model_name
        ],
        feature_view(
            dataset,
            model_name,
        )[mask],
    )

    probability[mask] = (
        channel_calibrators[
            model_name
        ].predict(score)
    )

    return probability


# =================================================================================================
# 16. EVENT-LEVEL AUDIT AND POLICY SELECTION
# =================================================================================================

def event_episode_audit(
    dataset: Dict,
    eligible_probability: np.ndarray,
    threshold: float,
) -> Dict:
    alert = (
        eligible_probability
        >= threshold
    ) & dataset[
        "hazard_eligible"
    ]

    total_events = 0
    detected_events = 0

    lead_times = []

    false_alert_samples = 0
    total_alert_samples = int(
        np.sum(alert)
    )

    false_alert_episodes = 0

    for stream_id in np.unique(
        dataset["stream_ids"]
    ):
        stream_mask = (
            dataset["stream_ids"]
            == stream_id
        )

        indices = np.flatnonzero(
            stream_mask
        )

        order = np.argsort(
            dataset[
                "row_times"
            ][indices]
        )

        indices = indices[order]

        stream_alert = alert[indices]
        stream_event = dataset[
            "y_hazard"
        ][indices].astype(bool)

        stream_times = dataset[
            "row_times"
        ][indices]

        event_indices = np.flatnonzero(
            stream_event
        )

        previous_event_time = None

        for local_event_index in event_indices:
            event_time = int(
                stream_times[
                    local_event_index
                ]
            )

            if (
                previous_event_time is not None
                and event_time
                <= previous_event_time
                + dataset["horizon"]
            ):
                continue

            total_events += 1

            window_start = max(
                0,
                local_event_index
                - dataset["horizon"]
                + 1,
            )

            candidate = np.flatnonzero(
                stream_alert[
                    window_start:
                    local_event_index + 1
                ]
            )

            if len(candidate) > 0:
                detected_events += 1

                alert_local_index = (
                    window_start
                    + candidate[0]
                )

                lead_times.append(
                    int(
                        event_time
                        - stream_times[
                            alert_local_index
                        ]
                        + 1
                    )
                )

            previous_event_time = event_time

        false_alert = (
            stream_alert
            & ~stream_event
        )

        false_alert_samples += int(
            np.sum(false_alert)
        )

        previous_false = False
        previous_time = None

        for time_value, is_false in zip(
            stream_times,
            false_alert,
        ):
            contiguous = bool(
                previous_time is not None
                and time_value
                == previous_time + 1
            )

            if (
                is_false
                and (
                    not previous_false
                    or not contiguous
                )
            ):
                false_alert_episodes += 1

            previous_false = bool(
                is_false
            )

            previous_time = int(
                time_value
            )

    eligible_count = int(
        np.sum(
            dataset[
                "hazard_eligible"
            ]
        )
    )

    return {
        "total_unique_event_onsets": int(
            total_events
        ),
        "detected_unique_event_onsets": int(
            detected_events
        ),
        "event_recall": float(
            detected_events
            / max(total_events, 1)
        ),
        "mean_detected_lead_time": (
            float(
                np.mean(lead_times)
            )
            if lead_times
            else None
        ),
        "median_detected_lead_time": (
            float(
                np.median(lead_times)
            )
            if lead_times
            else None
        ),
        "total_alert_samples": int(
            total_alert_samples
        ),
        "false_alert_samples": int(
            false_alert_samples
        ),
        "false_alert_fraction": float(
            false_alert_samples
            / max(
                total_alert_samples,
                1,
            )
        ),
        "false_alert_episodes": int(
            false_alert_episodes
        ),
        "false_alert_episodes_per_100_eligible_samples": float(
            100.0
            * false_alert_episodes
            / max(
                eligible_count,
                1,
            )
        ),
        "total_eligible_samples": int(
            eligible_count
        ),
    }


def select_event_policy_threshold(
    dataset: Dict,
    probability: np.ndarray,
) -> Dict:
    positive_probability = probability[
        dataset["hazard_eligible"]
    ]

    quantiles = np.linspace(
        0.02,
        0.98,
        120,
    )

    candidates = np.unique(
        np.concatenate([
            np.quantile(
                positive_probability,
                quantiles,
            ),
            np.asarray([
                0.01,
                0.025,
                0.05,
                0.10,
                0.20,
                0.50,
            ]),
        ])
    )

    best = None

    for threshold in candidates:
        audit = event_episode_audit(
            dataset,
            probability,
            float(threshold),
        )

        sample_metrics = evaluate_hazard(
            dataset[
                "y_hazard"
            ][
                dataset[
                    "hazard_eligible"
                ]
            ],
            probability[
                dataset[
                    "hazard_eligible"
                ]
            ],
            float(threshold),
        )

        utility = (
            2.0
            * audit["event_recall"]
            + 0.40
            * sample_metrics["precision"]
            - 0.06
            * audit[
                "false_alert_episodes_per_100_eligible_samples"
            ]
            - 0.20
            * sample_metrics["fpr"]
        )

        feasible = bool(
            sample_metrics["fpr"]
            <= CFG.maximum_policy_fpr
            and audit[
                "false_alert_episodes_per_100_eligible_samples"
            ]
            <= CFG.maximum_false_alert_episodes_per_100
        )

        result = {
            "threshold": float(
                threshold
            ),
            "utility": float(
                utility
            ),
            "feasible": feasible,
            "sample_metrics": (
                sample_metrics
            ),
            "event_audit": audit,
        }

        if best is None:
            best = result
            continue

        if (
            result["feasible"]
            and not best["feasible"]
        ):
            best = result
            continue

        if (
            result["feasible"]
            == best["feasible"]
            and result["utility"]
            > best["utility"]
        ):
            best = result

    return best


policy_probability = (
    calibrated_hazard_probability(
        policy_dataset,
        BEST_HAZARD_MODEL_NAME,
    )
)

threshold_selection = (
    select_event_policy_threshold(
        policy_dataset,
        policy_probability,
    )
)

HAZARD_DECISION_THRESHOLD = float(
    threshold_selection[
        "threshold"
    ]
)

print(
    "\nIndependent event-policy calibration:"
)

print(
    f"  Threshold             : "
    f"{HAZARD_DECISION_THRESHOLD:.4f}"
)

print(
    f"  Policy event recall   : "
    f"{threshold_selection['event_audit']['event_recall']:.4f}"
)

print(
    f"  False episodes / 100  : "
    f"{threshold_selection['event_audit']['false_alert_episodes_per_100_eligible_samples']:.4f}"
)

print(
    f"  Policy sample FPR     : "
    f"{threshold_selection['sample_metrics']['fpr']:.4f}"
)


# =================================================================================================
# 17. UNTOUCHED VALIDATION
# =================================================================================================

validation_point_prediction = (
    selected_point_model.predict(
        feature_view(
            validation_dataset,
            BEST_POINT_MODEL_NAME,
        )
    )
)

validation_point_metrics = (
    evaluate_point(
        validation_dataset[
            "y_point"
        ],
        validation_point_prediction,
        validation_dataset[
            "stream_ids"
        ],
    )
)

validation_hazard_probability = (
    calibrated_hazard_probability(
        validation_dataset,
        BEST_HAZARD_MODEL_NAME,
    )
)

validation_hazard_mask = (
    validation_dataset[
        "hazard_eligible"
    ]
)

validation_hazard_metrics = (
    evaluate_hazard(
        validation_dataset[
            "y_hazard"
        ][validation_hazard_mask],
        validation_hazard_probability[
            validation_hazard_mask
        ],
        HAZARD_DECISION_THRESHOLD,
    )
)

validation_oracle_point, validation_oracle_hazard = (
    structural_oracle(
        validation_dataset,
        CFG.oracle_rollouts_evaluation,
        CFG.seed + 30_000,
    )
)

validation_oracle_point_metrics = (
    evaluate_point(
        validation_dataset["y_point"],
        validation_oracle_point,
        validation_dataset["stream_ids"],
    )
)

validation_oracle_hazard_metrics = (
    evaluate_hazard(
        validation_dataset["y_hazard"][
            validation_hazard_mask
        ],
        validation_oracle_hazard[
            validation_hazard_mask
        ],
        threshold=0.5,
    )
)

validation_persistence_prediction = (
    validation_dataset["current_proxy"]
)

validation_persistence_metrics = (
    evaluate_point(
        validation_dataset["y_point"],
        validation_persistence_prediction,
        validation_dataset["stream_ids"],
    )
)

validation_constant_probability = np.full(
    np.sum(validation_hazard_mask),
    np.mean(
        validation_dataset["y_hazard"][
            validation_hazard_mask
        ]
    ),
    dtype=np.float64,
)

validation_constant_brier = float(
    brier_score_loss(
        validation_dataset["y_hazard"][
            validation_hazard_mask
        ],
        validation_constant_probability,
    )
)

validation_event_audit = (
    event_episode_audit(
        validation_dataset,
        validation_hazard_probability,
        HAZARD_DECISION_THRESHOLD,
    )
)

print_header(
    "🧪 TEST C — UNTOUCHED VALIDATION"
)

print("\nSelected causal point forecaster:")

print(
    f"  Model      : {BEST_POINT_MODEL_NAME}"
)

print(
    f"  rho        : "
    f"{validation_point_metrics['rho']:.4f}"
)

print(
    f"  Macro rho  : "
    f"{validation_point_metrics['macro_stream_rho']:.4f}"
)

print(
    f"  R²         : "
    f"{validation_point_metrics['r2']:.4f}"
)

print(
    f"  MAE        : "
    f"{validation_point_metrics['mae']:.4f}"
)

print("\nStructural Monte Carlo oracle:")

print(
    f"  rho        : "
    f"{validation_oracle_point_metrics['rho']:.4f}"
)

print(
    f"  Macro rho  : "
    f"{validation_oracle_point_metrics['macro_stream_rho']:.4f}"
)

print(
    f"  R²         : "
    f"{validation_oracle_point_metrics['r2']:.4f}"
)

print("\nObservable persistence baseline:")

print(
    f"  rho        : "
    f"{validation_persistence_metrics['rho']:.4f}"
)

print(
    f"  R²         : "
    f"{validation_persistence_metrics['r2']:.4f}"
)

print("\nCalibrated hazard forecaster:")

print(
    f"  Model      : {BEST_HAZARD_MODEL_NAME}"
)

print(
    f"  AUC        : "
    f"{(validation_hazard_metrics['auroc'] or 0.0):.4f}"
)

print(
    f"  AP         : "
    f"{(validation_hazard_metrics['auprc'] or 0.0):.4f}"
)

print(
    f"  Prevalence : "
    f"{validation_hazard_metrics['prevalence']:.4f}"
)

print(
    f"  Recall     : "
    f"{validation_hazard_metrics['recall']:.4f}"
)

print(
    f"  Precision  : "
    f"{validation_hazard_metrics['precision']:.4f}"
)

print(
    f"  FPR        : "
    f"{validation_hazard_metrics['fpr']:.4f}"
)

print(
    f"  Brier      : "
    f"{validation_hazard_metrics['brier']:.4f}"
)

print(
    f"  Const Brier: "
    f"{validation_constant_brier:.4f}"
)

print("\nStructural hazard oracle:")

print(
    f"  AUC        : "
    f"{(validation_oracle_hazard_metrics['auroc'] or 0.0):.4f}"
)

print(
    f"  AP         : "
    f"{(validation_oracle_hazard_metrics['auprc'] or 0.0):.4f}"
)

print(
    f"  Brier      : "
    f"{validation_oracle_hazard_metrics['brier']:.4f}"
)


# =================================================================================================
# 18. HORIZON-SPECIFIC STRUCTURAL AND LEARNED EVALUATION
# =================================================================================================

def fit_horizon_raw_ordered_models(
    horizon: int,
) -> Dict[str, Any]:
    horizon_datasets = (
        DATASETS_BY_HORIZON[horizon]
    )

    train = horizon_datasets[
        "TRAIN_MIXED_KNOWN"
    ]

    selection = horizon_datasets[
        "CALIBRATION_SELECTION"
    ]

    probability = horizon_datasets[
        "CALIBRATION_PROBABILITY"
    ]

    validation = horizon_datasets[
        "VALIDATION"
    ]

    point_model = ExtraTreesRegressor(
        n_estimators=320,
        min_samples_leaf=5,
        max_features=0.75,
        random_state=CFG.seed + horizon,
        n_jobs=-1,
    )

    point_model.fit(
        ordered_raw_features(train),
        train["y_point"],
    )

    selection_prediction = (
        point_model.predict(
            ordered_raw_features(
                selection
            )
        )
    )

    selection_metrics = (
        evaluate_point(
            selection["y_point"],
            selection_prediction,
            selection["stream_ids"],
        )
    )

    validation_prediction = (
        point_model.predict(
            ordered_raw_features(
                validation
            )
        )
    )

    validation_metrics = (
        evaluate_point(
            validation["y_point"],
            validation_prediction,
            validation["stream_ids"],
        )
    )

    train_mask = train[
        "hazard_eligible"
    ]

    probability_mask = probability[
        "hazard_eligible"
    ]

    validation_mask = validation[
        "hazard_eligible"
    ]

    hazard_model = ExtraTreesClassifier(
        n_estimators=360,
        min_samples_leaf=5,
        class_weight="balanced",
        max_features=0.75,
        random_state=CFG.seed + 100 + horizon,
        n_jobs=-1,
    )

    hazard_model.fit(
        ordered_raw_features(train)[
            train_mask
        ],
        train["y_hazard"][
            train_mask
        ],
    )

    probability_score = model_score(
        hazard_model,
        ordered_raw_features(
            probability
        )[probability_mask],
    )

    calibrator = PlattCalibrator()

    calibrator.fit(
        probability_score,
        probability["y_hazard"][
            probability_mask
        ],
    )

    validation_score = model_score(
        hazard_model,
        ordered_raw_features(
            validation
        )[validation_mask],
    )

    validation_probability = (
        calibrator.predict(
            validation_score
        )
    )

    validation_hazard_metrics_local = (
        evaluate_hazard(
            validation["y_hazard"][
                validation_mask
            ],
            validation_probability,
            threshold=0.5,
        )
    )

    oracle_point, oracle_hazard = (
        structural_oracle(
            validation,
            CFG.oracle_rollouts_evaluation,
            CFG.seed + 40_000 + horizon,
        )
    )

    oracle_point_metrics = (
        evaluate_point(
            validation["y_point"],
            oracle_point,
            validation["stream_ids"],
        )
    )

    oracle_hazard_metrics = (
        evaluate_hazard(
            validation["y_hazard"][
                validation_mask
            ],
            oracle_hazard[
                validation_mask
            ],
            threshold=0.5,
        )
    )

    return {
        "point_model": point_model,
        "hazard_model": hazard_model,
        "calibrator": calibrator,
        "selection_point": (
            selection_metrics
        ),
        "validation_point": (
            validation_metrics
        ),
        "validation_hazard": (
            validation_hazard_metrics_local
        ),
        "oracle_point": (
            oracle_point_metrics
        ),
        "oracle_hazard": (
            oracle_hazard_metrics
        ),
    }


print_header(
    "🧪 TEST D — HORIZON-SPECIFIC TEMPORAL VALIDATION"
)

horizon_results = {}

for horizon in CFG.forecast_horizons:
    result = (
        fit_horizon_raw_ordered_models(
            horizon
        )
    )

    horizon_results[
        horizon
    ] = {
        key: value
        for key, value in result.items()
        if key not in {
            "point_model",
            "hazard_model",
            "calibrator",
        }
    }

    learned_point = result[
        "validation_point"
    ]

    learned_hazard = result[
        "validation_hazard"
    ]

    oracle_point = result[
        "oracle_point"
    ]

    oracle_hazard = result[
        "oracle_hazard"
    ]

    print(
        f"H={horizon:<2} | "
        f"Learned point rho={learned_point['rho']:.4f} | "
        f"R²={learned_point['r2']:.4f} | "
        f"Oracle rho={oracle_point['rho']:.4f} | "
        f"Oracle R²={oracle_point['r2']:.4f} | "
        f"Learned hazard AUC="
        f"{(learned_hazard['auroc'] or 0.0):.4f} | "
        f"Oracle hazard AUC="
        f"{(oracle_hazard['auroc'] or 0.0):.4f}"
    )


# =================================================================================================
# 19. TEMPORAL ORDER ABLATIONS
# =================================================================================================

def transformed_temporal_dataset(
    dataset: Dict,
    mode: str,
    rng: Optional[
        np.random.Generator
    ] = None,
) -> Dict:
    transformed = dict(dataset)

    raw = np.array(
        dataset["X_raw"],
        copy=True,
    )

    activation = (
        np.array(
            dataset["X_activation"],
            copy=True,
        )
        if "X_activation" in dataset
        else None
    )

    if mode == "REVERSED":
        raw = raw[:, ::-1, :]

        if activation is not None:
            activation = (
                activation[:, ::-1, :]
            )

    elif mode == "SHUFFLED":
        if rng is None:
            raise ValueError(
                "SHUFFLED mode requires rng."
            )

        for row_index in range(
            len(raw)
        ):
            permutation = rng.permutation(
                raw.shape[1]
            )

            raw[row_index] = (
                raw[
                    row_index,
                    permutation,
                    :
                ]
            )

            if activation is not None:
                activation[row_index] = (
                    activation[
                        row_index,
                        permutation,
                        :
                    ]
                )

    elif mode == "CIRCULAR":
        if rng is None:
            raise ValueError(
                "CIRCULAR mode requires rng."
            )

        for row_index in range(
            len(raw)
        ):
            shift = int(
                rng.integers(
                    1,
                    raw.shape[1],
                )
            )

            raw[row_index] = np.roll(
                raw[row_index],
                shift,
                axis=0,
            )

            if activation is not None:
                activation[row_index] = (
                    np.roll(
                        activation[
                            row_index
                        ],
                        shift,
                        axis=0,
                    )
                )

    else:
        raise ValueError(
            f"Unknown transformation: {mode}"
        )

    transformed["X_raw"] = raw

    if activation is not None:
        transformed[
            "X_activation"
        ] = activation

    return transformed


def evaluate_order_variant(
    dataset: Dict,
) -> Tuple[float, float]:
    point_prediction = (
        selected_point_model.predict(
            feature_view(
                dataset,
                BEST_POINT_MODEL_NAME,
            )
        )
    )

    point_rho = safe_spearman(
        dataset["y_point"],
        point_prediction,
    )

    hazard_mask = dataset[
        "hazard_eligible"
    ]

    hazard_score = model_score(
        selected_hazard_model,
        feature_view(
            dataset,
            BEST_HAZARD_MODEL_NAME,
        )[hazard_mask],
    )

    hazard_probability = (
        channel_calibrators[
            BEST_HAZARD_MODEL_NAME
        ].predict(
            hazard_score
        )
    )

    hazard_auc = safe_auroc(
        dataset["y_hazard"][
            hazard_mask
        ],
        hazard_probability,
    )

    return (
        float(point_rho),
        float(
            hazard_auc
            if hazard_auc is not None
            else 0.5
        ),
    )


print_header(
    "🧪 TEST E — ORDER-IDENTIFIABLE TEMPORAL ABLATIONS"
)

ordered_point_rho, ordered_hazard_auc = (
    evaluate_order_variant(
        validation_dataset
    )
)

reversed_dataset = (
    transformed_temporal_dataset(
        validation_dataset,
        "REVERSED",
    )
)

reversed_point_rho, reversed_hazard_auc = (
    evaluate_order_variant(
        reversed_dataset
    )
)

shuffle_point_values = []
shuffle_hazard_values = []

circular_point_values = []
circular_hazard_values = []

order_rng = np.random.default_rng(
    CFG.seed + 50_000
)

for trial_index in range(
    CFG.order_shuffle_trials
):
    shuffled_dataset = (
        transformed_temporal_dataset(
            validation_dataset,
            "SHUFFLED",
            order_rng,
        )
    )

    shuffled_point, shuffled_hazard = (
        evaluate_order_variant(
            shuffled_dataset
        )
    )

    shuffle_point_values.append(
        shuffled_point
    )

    shuffle_hazard_values.append(
        shuffled_hazard
    )

    circular_dataset = (
        transformed_temporal_dataset(
            validation_dataset,
            "CIRCULAR",
            order_rng,
        )
    )

    circular_point, circular_hazard = (
        evaluate_order_variant(
            circular_dataset
        )
    )

    circular_point_values.append(
        circular_point
    )

    circular_hazard_values.append(
        circular_hazard
    )


point_shuffle_p95 = float(
    np.quantile(
        shuffle_point_values,
        0.95,
    )
)

hazard_shuffle_p95 = float(
    np.quantile(
        shuffle_hazard_values,
        0.95,
    )
)

point_circular_p95 = float(
    np.quantile(
        circular_point_values,
        0.95,
    )
)

hazard_circular_p95 = float(
    np.quantile(
        circular_hazard_values,
        0.95,
    )
)

point_order_null = max(
    reversed_point_rho,
    point_shuffle_p95,
    point_circular_p95,
)

hazard_order_null = max(
    reversed_hazard_auc,
    hazard_shuffle_p95,
    hazard_circular_p95,
)

point_order_advantage = float(
    ordered_point_rho
    - point_order_null
)

hazard_order_advantage = float(
    ordered_hazard_auc
    - hazard_order_null
)

print(
    f"Point  | Ordered={ordered_point_rho:.4f} | "
    f"Reversed={reversed_point_rho:.4f} | "
    f"Shuffle p95={point_shuffle_p95:.4f} | "
    f"Circular p95={point_circular_p95:.4f} | "
    f"Advantage={point_order_advantage:+.4f}"
)

print(
    f"Hazard | Ordered={ordered_hazard_auc:.4f} | "
    f"Reversed={reversed_hazard_auc:.4f} | "
    f"Shuffle p95={hazard_shuffle_p95:.4f} | "
    f"Circular p95={hazard_circular_p95:.4f} | "
    f"Advantage={hazard_order_advantage:+.4f}"
)


# =================================================================================================
# 20. EVENT-ONSET AND FALSE-ALERT AUDIT
# =================================================================================================

print_header(
    "🧪 TEST F — EVENT-ONSET AND FALSE-ALERT EPISODE AUDIT"
)

for key, value in (
    validation_event_audit.items()
):
    print(
        f"  {key:<55}: {value}"
    )


# =================================================================================================
# 21. CROSS-REGIME TRANSFER
# =================================================================================================

TRANSFER_GROUPS = (
    "AR2_TRANSFER",
    "DELAYED_TRANSFER",
    "OSCILLATORY_TRANSFER",
    "MEAN_REVERTING_TRANSFER",
    "ABRUPT_UNKNOWN",
    "MECHANISM_NOVEL",
)


def safe_recovery_fraction(
    model_score_value: float,
    persistence_score_value: float,
    oracle_score_value: float,
) -> float:
    denominator = (
        oracle_score_value
        - persistence_score_value
    )

    if abs(denominator) < 1e-8:
        return 0.0

    return float(
        (
            model_score_value
            - persistence_score_value
        )
        / denominator
    )


print_header(
    "🧪 TEST G — CROSS-REGIME TRANSFER"
)

transfer_results = {}

known_point_passes = 0
known_hazard_passes = 0

for group_offset, group_name in enumerate(
    TRANSFER_GROUPS
):
    dataset = DATASETS[
        group_name
    ]

    point_prediction = (
        selected_point_model.predict(
            feature_view(
                dataset,
                BEST_POINT_MODEL_NAME,
            )
        )
    )

    point_metrics = evaluate_point(
        dataset["y_point"],
        point_prediction,
        dataset["stream_ids"],
    )

    persistence_metrics = evaluate_point(
        dataset["y_point"],
        dataset["current_proxy"],
        dataset["stream_ids"],
    )

    oracle_point, oracle_hazard = (
        structural_oracle(
            dataset,
            CFG.oracle_rollouts_evaluation,
            CFG.seed
            + 60_000
            + group_offset,
        )
    )

    oracle_point_metrics = (
        evaluate_point(
            dataset["y_point"],
            oracle_point,
            dataset["stream_ids"],
        )
    )

    point_recovery = (
        safe_recovery_fraction(
            point_metrics["rho"],
            persistence_metrics["rho"],
            oracle_point_metrics["rho"],
        )
    )

    hazard_probability = (
        calibrated_hazard_probability(
            dataset,
            BEST_HAZARD_MODEL_NAME,
        )
    )

    hazard_mask = dataset[
        "hazard_eligible"
    ]

    hazard_metrics = (
        evaluate_hazard(
            dataset["y_hazard"][
                hazard_mask
            ],
            hazard_probability[
                hazard_mask
            ],
            HAZARD_DECISION_THRESHOLD,
        )
    )

    oracle_hazard_metrics = (
        evaluate_hazard(
            dataset["y_hazard"][
                hazard_mask
            ],
            oracle_hazard[
                hazard_mask
            ],
            threshold=0.5,
        )
    )

    hazard_auc = (
        hazard_metrics["auroc"]
        or 0.0
    )

    hazard_ap = (
        hazard_metrics["auprc"]
        or 0.0
    )

    hazard_ap_gain = float(
        hazard_ap
        - hazard_metrics[
            "prevalence"
        ]
    )

    known_family = (
        group_name
        in {
            "AR2_TRANSFER",
            "DELAYED_TRANSFER",
            "OSCILLATORY_TRANSFER",
            "MEAN_REVERTING_TRANSFER",
        }
    )

    oracle_forecastable = bool(
        oracle_point_metrics["rho"]
        >= CFG.minimum_preflight_point_rho
        and oracle_point_metrics["r2"]
        >= CFG.minimum_preflight_point_r2
    )

    point_gate = bool(
        oracle_forecastable
        and point_metrics["rho"]
        >= CFG.minimum_point_rho
        and point_metrics[
            "macro_stream_rho"
        ]
        >= CFG.minimum_point_macro_rho
        and point_metrics["r2"]
        >= CFG.minimum_point_r2
        and point_metrics["rho"]
        > persistence_metrics["rho"]
    )

    hazard_oracle_auc = (
        oracle_hazard_metrics[
            "auroc"
        ]
        or 0.0
    )

    hazard_oracle_forecastable = bool(
        hazard_oracle_auc
        >= CFG.minimum_preflight_hazard_auc
    )

    hazard_gate = bool(
        hazard_oracle_forecastable
        and hazard_auc
        >= CFG.minimum_hazard_auc
        and hazard_ap_gain
        >= CFG.minimum_hazard_ap_gain
        and hazard_metrics["fpr"]
        <= CFG.maximum_policy_fpr
    )

    if known_family:
        known_point_passes += int(
            point_gate
        )

        known_hazard_passes += int(
            hazard_gate
        )

    transfer_results[
        group_name
    ] = {
        "point": point_metrics,
        "persistence": (
            persistence_metrics
        ),
        "point_oracle": (
            oracle_point_metrics
        ),
        "point_recovery": (
            point_recovery
        ),
        "point_oracle_forecastable": (
            oracle_forecastable
        ),
        "point_gate": point_gate,
        "hazard": hazard_metrics,
        "hazard_oracle": (
            oracle_hazard_metrics
        ),
        "hazard_ap_gain": (
            hazard_ap_gain
        ),
        "hazard_oracle_forecastable": (
            hazard_oracle_forecastable
        ),
        "hazard_gate": hazard_gate,
    }

    print(f"\n{group_name}")

    print(
        f"  Point  | "
        f"rho={point_metrics['rho']:.4f} | "
        f"Macro={point_metrics['macro_stream_rho']:.4f} | "
        f"R²={point_metrics['r2']:.4f} | "
        f"Persist={persistence_metrics['rho']:.4f} | "
        f"Oracle={oracle_point_metrics['rho']:.4f} | "
        f"Recovery={point_recovery:.4f} | "
        f"Gate={'PASS' if point_gate else 'FAIL'}"
    )

    print(
        f"  Hazard | "
        f"AUC={hazard_auc:.4f} | "
        f"AP={hazard_ap:.4f} | "
        f"Prev={hazard_metrics['prevalence']:.4f} | "
        f"ΔAP={hazard_ap_gain:+.4f} | "
        f"FPR={hazard_metrics['fpr']:.4f} | "
        f"Oracle AUC={hazard_oracle_auc:.4f} | "
        f"Gate={'PASS' if hazard_gate else 'FAIL'}"
    )


# =================================================================================================
# 22. TRANSITION-RESIDUAL NOVELTY SENSOR
# =================================================================================================

def transition_prediction_features(
    dataset: Dict,
) -> np.ndarray:
    raw = dataset["X_raw"]
    activation = dataset[
        "X_activation"
    ]

    raw_past = raw[
        :,
        :-1,
        :
    ].reshape(
        len(raw),
        -1,
    )

    latest_context = raw[
        :,
        -2,
        :
    ]

    activation_past = activation[
        :,
        :-1,
        :
    ]

    activation_summary = (
        np.concatenate([
            np.mean(
                activation_past,
                axis=1,
            ),
            np.std(
                activation_past,
                axis=1,
            ),
            activation_past[
                :,
                -1,
                :
            ],
        ], axis=1)
    )

    return np.concatenate([
        raw_past,
        latest_context,
        activation_summary,
    ], axis=1)


def transition_target(
    dataset: Dict,
) -> np.ndarray:
    return dataset[
        "X_raw"
    ][:, -1, 4]


transition_model = (
    ExtraTreesRegressor(
        n_estimators=420,
        min_samples_leaf=5,
        max_features=0.72,
        random_state=CFG.seed + 70_000,
        n_jobs=-1,
    )
)

transition_model.fit(
    transition_prediction_features(
        train_dataset
    ),
    transition_target(
        train_dataset
    ),
)


def raw_transition_residual_score(
    dataset: Dict,
) -> np.ndarray:
    prediction = (
        transition_model.predict(
            transition_prediction_features(
                dataset
            )
        )
    )

    target = transition_target(
        dataset
    )

    return np.abs(
        target
        - prediction
    )


selection_transition_score = (
    raw_transition_residual_score(
        selection_dataset
    )
)

transition_center = float(
    np.median(
        selection_transition_score
    )
)

transition_scale = float(
    1.4826
    * np.median(
        np.abs(
            selection_transition_score
            - transition_center
        )
    )
    + 1e-6
)


def activation_novelty_features(
    dataset: Dict,
) -> np.ndarray:
    activation = dataset[
        "X_activation"
    ]

    return np.concatenate([
        activation[:, -1, :],
        activation[:, -1, :]
        - activation[:, -2, :],
        np.mean(
            activation,
            axis=1,
        ),
        np.std(
            activation,
            axis=1,
        ),
    ], axis=1)


activation_novelty_model = (
    IsolationForest(
        n_estimators=360,
        max_samples="auto",
        contamination="auto",
        random_state=CFG.seed + 71_000,
        n_jobs=-1,
    )
)

activation_novelty_model.fit(
    activation_novelty_features(
        train_dataset
    )
)

selection_activation_raw_score = (
    -activation_novelty_model.score_samples(
        activation_novelty_features(
            selection_dataset
        )
    )
)

activation_score_center = float(
    np.median(
        selection_activation_raw_score
    )
)

activation_score_scale = float(
    1.4826
    * np.median(
        np.abs(
            selection_activation_raw_score
            - activation_score_center
        )
    )
    + 1e-6
)


def mechanism_novelty_score(
    dataset: Dict,
) -> np.ndarray:
    transition_score = (
        raw_transition_residual_score(
            dataset
        )
    )

    transition_z = (
        transition_score
        - transition_center
    ) / transition_scale

    activation_score = (
        -activation_novelty_model.score_samples(
            activation_novelty_features(
                dataset
            )
        )
    )

    activation_z = (
        activation_score
        - activation_score_center
    ) / activation_score_scale

    combined_score = (
        0.75 * transition_z
        + 0.25 * activation_z
    )

    return combined_score.astype(
        np.float64
    )


novelty_policy_score = (
    mechanism_novelty_score(
        policy_dataset
    )
)

NOVELTY_THRESHOLD = float(
    np.quantile(
        novelty_policy_score,
        1.0
        - CFG.maximum_novelty_fpr,
    )
)

validation_novelty_score = (
    mechanism_novelty_score(
        validation_dataset
    )
)

validation_novelty_fpr = float(
    np.mean(
        validation_novelty_score
        >= NOVELTY_THRESHOLD
    )
)

known_transfer_scores = []
known_transfer_labels = []

for group_name in (
    "AR2_TRANSFER",
    "DELAYED_TRANSFER",
    "OSCILLATORY_TRANSFER",
    "MEAN_REVERTING_TRANSFER",
):
    group_score = (
        mechanism_novelty_score(
            DATASETS[group_name]
        )
    )

    known_transfer_scores.append(
        group_score
    )

    known_transfer_labels.append(
        np.zeros(
            len(group_score),
            dtype=np.int64,
        )
    )

known_transfer_score = (
    np.concatenate(
        known_transfer_scores
    )
)

known_transfer_novelty_fpr = float(
    np.mean(
        known_transfer_score
        >= NOVELTY_THRESHOLD
    )
)

unknown_group_scores = {}

for group_name in (
    "ABRUPT_UNKNOWN",
    "MECHANISM_NOVEL",
):
    unknown_group_scores[
        group_name
    ] = mechanism_novelty_score(
        DATASETS[group_name]
    )

unknown_score = np.concatenate([
    unknown_group_scores[
        "ABRUPT_UNKNOWN"
    ],
    unknown_group_scores[
        "MECHANISM_NOVEL"
    ],
])

unknown_novelty_tpr = float(
    np.mean(
        unknown_score
        >= NOVELTY_THRESHOLD
    )
)

novelty_evaluation_labels = (
    np.concatenate([
        np.zeros(
            len(known_transfer_score),
            dtype=np.int64,
        ),
        np.ones(
            len(unknown_score),
            dtype=np.int64,
        ),
    ])
)

novelty_evaluation_scores = (
    np.concatenate([
        known_transfer_score,
        unknown_score,
    ])
)

novelty_auc = safe_auroc(
    novelty_evaluation_labels,
    novelty_evaluation_scores,
)

novelty_ap = safe_auprc(
    novelty_evaluation_labels,
    novelty_evaluation_scores,
)

print_header(
    "🧪 TEST H — MATCHED TRANSITION-RESIDUAL MECHANISM NOVELTY"
)

print(
    f"Novelty AUROC                 : "
    f"{(novelty_auc or 0.0):.4f}"
)

print(
    f"Novelty AUPRC                 : "
    f"{(novelty_ap or 0.0):.4f}"
)

print(
    f"Validation known FPR          : "
    f"{validation_novelty_fpr:.4f}"
)

print(
    f"Known-parameter-transfer FPR  : "
    f"{known_transfer_novelty_fpr:.4f}"
)

print(
    f"Unknown-mechanism TPR         : "
    f"{unknown_novelty_tpr:.4f}"
)

print(
    f"Novelty threshold             : "
    f"{NOVELTY_THRESHOLD:.4f}"
)

for group_name, score in (
    unknown_group_scores.items()
):
    print(
        f"  {group_name:<22} | "
        f"mean score={np.mean(score):.4f} | "
        f"TPR={np.mean(score >= NOVELTY_THRESHOLD):.4f}"
    )


# =================================================================================================
# 23. NOVELTY / DEGRADATION CONFOUNDING AUDIT
# =================================================================================================

def binary_phi(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    first = np.asarray(
        first,
        dtype=np.int64,
    )

    second = np.asarray(
        second,
        dtype=np.int64,
    )

    if (
        len(first) < 2
        or len(np.unique(first)) < 2
        or len(np.unique(second)) < 2
    ):
        return 0.0

    correlation = np.corrcoef(
        first,
        second,
    )[0, 1]

    if not np.isfinite(
        correlation
    ):
        return 0.0

    return float(
        correlation
    )


known_for_confounding = np.concatenate([
    DATASETS["AR2_TRANSFER"][
        "y_point"
    ],
    DATASETS["DELAYED_TRANSFER"][
        "y_point"
    ],
    DATASETS["OSCILLATORY_TRANSFER"][
        "y_point"
    ],
    DATASETS["MEAN_REVERTING_TRANSFER"][
        "y_point"
    ],
])

unknown_for_confounding = np.concatenate([
    DATASETS["ABRUPT_UNKNOWN"][
        "y_point"
    ],
    DATASETS["MECHANISM_NOVEL"][
        "y_point"
    ],
])

generator_novelty_labels = np.concatenate([
    np.zeros(
        len(known_for_confounding),
        dtype=np.int64,
    ),
    np.ones(
        len(unknown_for_confounding),
        dtype=np.int64,
    ),
])

generator_degradation_labels = (
    np.abs(
        np.concatenate([
            known_for_confounding,
            unknown_for_confounding,
        ])
    )
    >= CFG.latent_event_threshold
).astype(np.int64)

generator_novelty_degradation_phi = (
    binary_phi(
        generator_novelty_labels,
        generator_degradation_labels,
    )
)

detector_labels = (
    novelty_evaluation_scores
    >= NOVELTY_THRESHOLD
).astype(np.int64)

detector_degradation_labels = (
    np.abs(
        np.concatenate([
            known_for_confounding,
            unknown_for_confounding,
        ])
    )
    >= CFG.latent_event_threshold
).astype(np.int64)

minimum_length = min(
    len(detector_labels),
    len(detector_degradation_labels),
)

detector_degradation_phi = (
    binary_phi(
        detector_labels[
            :minimum_length
        ],
        detector_degradation_labels[
            :minimum_length
        ],
    )
)

print_header(
    "🧪 TEST I — NOVELTY / DEGRADATION DISENTANGLEMENT"
)

print(
    "Generator novelty/degradation phi : "
    f"{generator_novelty_degradation_phi:+.4f}"
)

print(
    "Detector/degradation phi           : "
    f"{detector_degradation_phi:+.4f}"
)


# =================================================================================================
# 24. INDEPENDENT RAW VS ACTIVATION-COMBINED HAZARD AUDIT
# =================================================================================================

combined_policy_probability = (
    calibrated_hazard_probability(
        policy_dataset,
        "COMBINED_ORDERED",
    )
)

combined_threshold_selection = (
    select_event_policy_threshold(
        policy_dataset,
        combined_policy_probability,
    )
)

COMBINED_HAZARD_THRESHOLD = float(
    combined_threshold_selection[
        "threshold"
    ]
)

channel_separation_results = {}

for group_name in (
    "ABRUPT_UNKNOWN",
    "MECHANISM_NOVEL",
):
    dataset = DATASETS[
        group_name
    ]

    hazard_mask = dataset[
        "hazard_eligible"
    ]

    raw_probability = (
        calibrated_hazard_probability(
            dataset,
            BEST_HAZARD_MODEL_NAME,
        )
    )

    combined_probability = (
        calibrated_hazard_probability(
            dataset,
            "COMBINED_ORDERED",
        )
    )

    raw_metrics = (
        evaluate_hazard(
            dataset["y_hazard"][
                hazard_mask
            ],
            raw_probability[
                hazard_mask
            ],
            HAZARD_DECISION_THRESHOLD,
        )
    )

    combined_metrics = (
        evaluate_hazard(
            dataset["y_hazard"][
                hazard_mask
            ],
            combined_probability[
                hazard_mask
            ],
            COMBINED_HAZARD_THRESHOLD,
        )
    )

    channel_separation_results[
        group_name
    ] = {
        "raw_causal": raw_metrics,
        "combined_diagnostic": (
            combined_metrics
        ),
        "raw_threshold": (
            HAZARD_DECISION_THRESHOLD
        ),
        "combined_threshold": (
            COMBINED_HAZARD_THRESHOLD
        ),
        "delta_fpr": float(
            combined_metrics["fpr"]
            - raw_metrics["fpr"]
        ),
        "delta_brier": float(
            combined_metrics["brier"]
            - raw_metrics["brier"]
        ),
    }

    print(f"\n{group_name}")

    print(
        f"  Raw causal | "
        f"AUC={(raw_metrics['auroc'] or 0.0):.4f} | "
        f"AP={(raw_metrics['auprc'] or 0.0):.4f} | "
        f"FPR={raw_metrics['fpr']:.4f} | "
        f"Brier={raw_metrics['brier']:.4f}"
    )

    print(
        f"  Combined   | "
        f"AUC={(combined_metrics['auroc'] or 0.0):.4f} | "
        f"AP={(combined_metrics['auprc'] or 0.0):.4f} | "
        f"FPR={combined_metrics['fpr']:.4f} | "
        f"Brier={combined_metrics['brier']:.4f}"
    )

    print(
        f"  ΔFPR combined−raw   : "
        f"{combined_metrics['fpr'] - raw_metrics['fpr']:+.4f}"
    )

    print(
        f"  ΔBrier combined−raw : "
        f"{combined_metrics['brier'] - raw_metrics['brier']:+.4f}"
    )


# =================================================================================================
# 25. STREAM-LEVEL BOOTSTRAP INTERVALS
# =================================================================================================

def percentile_interval(
    values: Sequence[float],
    lower: float = 0.025,
    upper: float = 0.975,
) -> List[float]:
    valid = np.asarray([
        value
        for value in values
        if value is not None
        and np.isfinite(value)
    ], dtype=np.float64)

    if len(valid) == 0:
        return [
            float("nan"),
            float("nan"),
        ]

    return [
        float(
            np.quantile(
                valid,
                lower,
            )
        ),
        float(
            np.quantile(
                valid,
                upper,
            )
        ),
    ]


def stream_bootstrap_intervals(
    dataset: Dict,
    point_prediction: np.ndarray,
    hazard_probability: np.ndarray,
    iterations: int,
    seed: int,
) -> Dict:
    rng = np.random.default_rng(
        seed
    )

    unique_streams = np.unique(
        dataset["stream_ids"]
    )

    point_rhos = []
    point_macro_rhos = []
    point_r2_values = []

    hazard_aucs = []
    hazard_aps = []
    hazard_briers = []

    for _ in range(
        iterations
    ):
        sampled_streams = rng.choice(
            unique_streams,
            size=len(unique_streams),
            replace=True,
        )

        sampled_indices = []
        resampled_ids = []

        for bootstrap_stream_id, original_stream_id in enumerate(
            sampled_streams
        ):
            indices = np.flatnonzero(
                dataset["stream_ids"]
                == original_stream_id
            )

            sampled_indices.append(
                indices
            )

            resampled_ids.append(
                np.full(
                    len(indices),
                    bootstrap_stream_id,
                    dtype=np.int64,
                )
            )

        sampled_indices = (
            np.concatenate(
                sampled_indices
            )
        )

        resampled_ids = (
            np.concatenate(
                resampled_ids
            )
        )

        point_metrics = evaluate_point(
            dataset["y_point"][
                sampled_indices
            ],
            point_prediction[
                sampled_indices
            ],
            resampled_ids,
        )

        point_rhos.append(
            point_metrics["rho"]
        )

        point_macro_rhos.append(
            point_metrics[
                "macro_stream_rho"
            ]
        )

        point_r2_values.append(
            point_metrics["r2"]
        )

        eligible = dataset[
            "hazard_eligible"
        ][sampled_indices]

        labels = dataset[
            "y_hazard"
        ][sampled_indices][eligible]

        probability = hazard_probability[
            sampled_indices
        ][eligible]

        if len(np.unique(labels)) < 2:
            continue

        hazard_aucs.append(
            roc_auc_score(
                labels,
                probability,
            )
        )

        hazard_aps.append(
            average_precision_score(
                labels,
                probability,
            )
        )

        hazard_briers.append(
            brier_score_loss(
                labels,
                probability,
            )
        )

    return {
        "point_rho_95_ci": (
            percentile_interval(
                point_rhos
            )
        ),
        "point_macro_rho_95_ci": (
            percentile_interval(
                point_macro_rhos
            )
        ),
        "point_r2_95_ci": (
            percentile_interval(
                point_r2_values
            )
        ),
        "hazard_auc_95_ci": (
            percentile_interval(
                hazard_aucs
            )
        ),
        "hazard_ap_95_ci": (
            percentile_interval(
                hazard_aps
            )
        ),
        "hazard_brier_95_ci": (
            percentile_interval(
                hazard_briers
            )
        ),
    }


bootstrap_intervals = (
    stream_bootstrap_intervals(
        validation_dataset,
        validation_point_prediction,
        validation_hazard_probability,
        CFG.bootstrap_iterations,
        CFG.seed + 80_000,
    )
)

print_header(
    "🧪 TEST J — STREAM-LEVEL BOOTSTRAP INTERVALS"
)

print(
    "Point rho 95% CI       : "
    f"{bootstrap_intervals['point_rho_95_ci']}"
)

print(
    "Point macro rho 95% CI : "
    f"{bootstrap_intervals['point_macro_rho_95_ci']}"
)

print(
    "Point R² 95% CI        : "
    f"{bootstrap_intervals['point_r2_95_ci']}"
)

print(
    "Hazard AUC 95% CI      : "
    f"{bootstrap_intervals['hazard_auc_95_ci']}"
)

print(
    "Hazard AP 95% CI       : "
    f"{bootstrap_intervals['hazard_ap_95_ci']}"
)

print(
    "Hazard Brier 95% CI    : "
    f"{bootstrap_intervals['hazard_brier_95_ci']}"
)


# =================================================================================================
# 26. AUTOMATED SCIENTIFIC GATES
# =================================================================================================

feedback_observability_valid = bool(
    layer_observability_results[
        BEST_FEEDBACK_LAYER
    ]["rho"]
    >= CFG.minimum_observability_rho
    and layer_observability_results[
        BEST_FEEDBACK_LAYER
    ]["r2"]
    >= CFG.minimum_observability_r2
)

point_oracle_forecastable = bool(
    validation_oracle_point_metrics[
        "rho"
    ]
    >= CFG.minimum_preflight_point_rho
    and validation_oracle_point_metrics[
        "r2"
    ]
    >= CFG.minimum_preflight_point_r2
)

hazard_oracle_forecastable = bool(
    (
        validation_oracle_hazard_metrics[
            "auroc"
        ]
        or 0.0
    )
    >= CFG.minimum_preflight_hazard_auc
)

selected_point_forecaster_valid = bool(
    validation_point_metrics["rho"]
    >= CFG.minimum_point_rho
    and validation_point_metrics[
        "macro_stream_rho"
    ]
    >= CFG.minimum_point_macro_rho
    and validation_point_metrics["r2"]
    >= CFG.minimum_point_r2
    and validation_point_metrics["rho"]
    > validation_persistence_metrics[
        "rho"
    ]
)

validation_hazard_ap_gain = float(
    (
        validation_hazard_metrics[
            "auprc"
        ]
        or 0.0
    )
    - validation_hazard_metrics[
        "prevalence"
    ]
)

selected_hazard_forecaster_valid = bool(
    (
        validation_hazard_metrics[
            "auroc"
        ]
        or 0.0
    )
    >= CFG.minimum_hazard_auc
    and validation_hazard_ap_gain
    >= CFG.minimum_hazard_ap_gain
    and validation_hazard_metrics[
        "brier"
    ]
    < validation_constant_brier
)

point_temporal_order_established = bool(
    BEST_POINT_MODEL_NAME
    in {
        "RAW_ORDERED",
    }
    and point_order_advantage
    >= CFG.minimum_order_advantage
)

hazard_temporal_order_established = bool(
    BEST_HAZARD_MODEL_NAME
    in {
        "RAW_ORDERED",
    }
    and hazard_order_advantage
    >= CFG.minimum_order_advantage
)

probability_calibration_valid = bool(
    validation_hazard_metrics[
        "brier"
    ]
    < validation_constant_brier
)

operational_warning_policy_valid = bool(
    validation_event_audit[
        "event_recall"
    ]
    >= CFG.minimum_event_recall
    and validation_event_audit[
        "false_alert_episodes_per_100_eligible_samples"
    ]
    <= CFG.maximum_false_alert_episodes_per_100
    and validation_hazard_metrics[
        "fpr"
    ]
    <= CFG.maximum_policy_fpr
)

novelty_sensor_valid = bool(
    (
        novelty_auc
        or 0.0
    )
    >= CFG.minimum_novelty_auc
    and unknown_novelty_tpr
    >= CFG.minimum_novelty_tpr
    and validation_novelty_fpr
    <= CFG.maximum_novelty_fpr
    and known_transfer_novelty_fpr
    <= CFG.maximum_novelty_fpr
)

factorial_novelty_independence_valid = bool(
    abs(
        generator_novelty_degradation_phi
    )
    <= 0.20
)

detector_novelty_independence_valid = bool(
    abs(
        detector_degradation_phi
    )
    <= 0.25
)

causal_novelty_channel_separation_valid = bool(
    BEST_POINT_MODEL_NAME
    in CAUSAL_MODEL_NAMES
    and BEST_HAZARD_MODEL_NAME
    in CAUSAL_MODEL_NAMES
)

point_transfer_strict_majority = bool(
    known_point_passes
    >= 3
)

hazard_transfer_strict_majority = bool(
    known_hazard_passes
    >= 3
)

benchmark_construction_valid = bool(
    primary_preflight_passed
    and point_oracle_forecastable
    and hazard_oracle_forecastable
    and factorial_novelty_independence_valid
)

observable_feedback_self_model_valid = (
    feedback_observability_valid
)

temporal_order_learning_valid = bool(
    selected_point_forecaster_valid
    and selected_hazard_forecaster_valid
    and point_temporal_order_established
    and hazard_temporal_order_established
)

hazard_event_onset_system_valid = bool(
    selected_hazard_forecaster_valid
    and probability_calibration_valid
    and operational_warning_policy_valid
)

cross_regime_transfer_valid = bool(
    point_transfer_strict_majority
    and hazard_transfer_strict_majority
)

disentangled_architecture_valid = bool(
    novelty_sensor_valid
    and factorial_novelty_independence_valid
    and detector_novelty_independence_valid
    and causal_novelty_channel_separation_valid
)

overall_valid = bool(
    benchmark_construction_valid
    and observable_feedback_self_model_valid
    and temporal_order_learning_valid
    and hazard_event_onset_system_valid
    and cross_regime_transfer_valid
    and novelty_sensor_valid
    and disentangled_architecture_valid
)


scientific_gates = {
    "feedback_observability_valid": (
        feedback_observability_valid
    ),
    "point_oracle_forecastable": (
        point_oracle_forecastable
    ),
    "hazard_oracle_forecastable": (
        hazard_oracle_forecastable
    ),
    "selected_point_forecaster_valid": (
        selected_point_forecaster_valid
    ),
    "selected_hazard_forecaster_valid": (
        selected_hazard_forecaster_valid
    ),
    "point_temporal_order_established": (
        point_temporal_order_established
    ),
    "hazard_temporal_order_established": (
        hazard_temporal_order_established
    ),
    "probability_calibration_valid": (
        probability_calibration_valid
    ),
    "operational_warning_policy_valid": (
        operational_warning_policy_valid
    ),
    "novelty_sensor_valid": (
        novelty_sensor_valid
    ),
    "factorial_novelty_independence_valid": (
        factorial_novelty_independence_valid
    ),
    "detector_novelty_independence_valid": (
        detector_novelty_independence_valid
    ),
    "causal_novelty_channel_separation_valid": (
        causal_novelty_channel_separation_valid
    ),
    "point_transfer_strict_majority": (
        point_transfer_strict_majority
    ),
    "hazard_transfer_strict_majority": (
        hazard_transfer_strict_majority
    ),
}


overall_statuses = {
    "benchmark_construction": (
        benchmark_construction_valid
    ),
    "observable_feedback_self_model": (
        observable_feedback_self_model_valid
    ),
    "temporal_order_learning": (
        temporal_order_learning_valid
    ),
    "hazard_event_onset_system": (
        hazard_event_onset_system_valid
    ),
    "cross_regime_transfer": (
        cross_regime_transfer_valid
    ),
    "novelty_detection": (
        novelty_sensor_valid
    ),
    "disentangled_architecture": (
        disentangled_architecture_valid
    ),
    "overall": overall_valid,
}


diagnostic_messages = []

diagnostic_map = {
    "feedback_observability_valid": (
        "ACTIVATION_FEEDBACK_OBSERVABILITY_FAILED"
    ),
    "point_oracle_forecastable": (
        "POINT_TARGET_NOT_FORECASTABLE_BY_STRUCTURAL_ORACLE"
    ),
    "hazard_oracle_forecastable": (
        "HAZARD_TARGET_NOT_FORECASTABLE_BY_STRUCTURAL_ORACLE"
    ),
    "selected_point_forecaster_valid": (
        "SELECTED_POINT_FORECASTER_INVALID"
    ),
    "selected_hazard_forecaster_valid": (
        "SELECTED_HAZARD_FORECASTER_INVALID"
    ),
    "point_temporal_order_established": (
        "POINT_TEMPORAL_ORDER_NOT_ESTABLISHED"
    ),
    "hazard_temporal_order_established": (
        "HAZARD_TEMPORAL_ORDER_NOT_ESTABLISHED"
    ),
    "probability_calibration_valid": (
        "HAZARD_PROBABILITY_CALIBRATION_FAILED"
    ),
    "operational_warning_policy_valid": (
        "OPERATIONAL_WARNING_POLICY_FAILED"
    ),
    "novelty_sensor_valid": (
        "MECHANISM_NOVELTY_SENSOR_FAILED"
    ),
    "factorial_novelty_independence_valid": (
        "GENERATOR_NOVELTY_DEGRADATION_CONFOUNDING"
    ),
    "detector_novelty_independence_valid": (
        "DETECTOR_NOVELTY_DEGRADATION_CONFOUNDING"
    ),
    "causal_novelty_channel_separation_valid": (
        "ACTIVATION_CONTAMINATED_CAUSAL_CHANNEL_SELECTION"
    ),
    "point_transfer_strict_majority": (
        "POINT_KNOWN_FAMILY_TRANSFER_FAILED"
    ),
    "hazard_transfer_strict_majority": (
        "HAZARD_KNOWN_FAMILY_TRANSFER_FAILED"
    ),
}

for gate_name, gate_value in (
    scientific_gates.items()
):
    if not gate_value:
        diagnostic_messages.append(
            diagnostic_map[
                gate_name
            ]
        )

if not diagnostic_messages:
    diagnostic_messages.append(
        "ALL_PRIMARY_SCIENTIFIC_GATES_PASSED"
    )


# =================================================================================================
# 27. SUMMARY
# =================================================================================================

print_header(
    "🧠 AUTOMATED DIAGNOSTIC SUMMARY"
)

for gate_name, gate_value in (
    scientific_gates.items()
):
    print(
        f"{'✅' if gate_value else '❌'} "
        f"{gate_name}"
    )

print(
    "\nKnown-family point transfer gates:"
)

for group_name in (
    "AR2_TRANSFER",
    "DELAYED_TRANSFER",
    "OSCILLATORY_TRANSFER",
    "MEAN_REVERTING_TRANSFER",
):
    passed = transfer_results[
        group_name
    ]["point_gate"]

    print(
        f"  {'✅' if passed else '❌'} "
        f"{group_name}"
    )

print(
    f"\nPoint transfer count: "
    f"{known_point_passes}/4 "
    f"(strict majority requires 3)"
)

print(
    "\nKnown-family hazard transfer gates:"
)

for group_name in (
    "AR2_TRANSFER",
    "DELAYED_TRANSFER",
    "OSCILLATORY_TRANSFER",
    "MEAN_REVERTING_TRANSFER",
):
    passed = transfer_results[
        group_name
    ]["hazard_gate"]

    print(
        f"  {'✅' if passed else '❌'} "
        f"{group_name}"
    )

print(
    f"\nHazard transfer count: "
    f"{known_hazard_passes}/4 "
    f"(strict majority requires 3)"
)

print("\nSelected architecture:")

print(
    f"  Feedback layer        : "
    f"{BEST_FEEDBACK_LAYER}"
)

print(
    f"  Activation PCA dim    : "
    f"{CFG.activation_pca_dimension}"
)

print(
    f"  Causal point model    : "
    f"{BEST_POINT_MODEL_NAME}"
)

print(
    f"  Causal hazard model   : "
    f"{BEST_HAZARD_MODEL_NAME}"
)

print(
    f"  Hazard threshold      : "
    f"{HAZARD_DECISION_THRESHOLD:.4f}"
)

print(
    f"  Combined threshold    : "
    f"{COMBINED_HAZARD_THRESHOLD:.4f}"
)

print(
    f"  Novelty threshold     : "
    f"{NOVELTY_THRESHOLD:.4f}"
)

print(
    "  Architecture policy  : "
    "RAW causal forecasting + "
    "transition-residual/activation novelty sensing"
)

print("\nOverall scientific statuses:")

for status_name, status_value in (
    overall_statuses.items()
):
    print(
        f"  {status_name:<38}: "
        f"{'PASS' if status_value else 'FAIL'}"
    )

print("\nFinal diagnosis:")

print(
    " | ".join(
        diagnostic_messages
    )
)


# =================================================================================================
# 28. TELEMETRY SERIALIZATION
# =================================================================================================

telemetry = {
    "milestone": CFG.milestone,
    "title": CFG.title,
    "execution_status": "COMPLETE",
    "configuration": asdict(CFG),
    "device": DEVICE,
    "model": CFG.model_name,
    "valid_layers": valid_layers,
    "selected_architecture": {
        "feedback_layer": (
            BEST_FEEDBACK_LAYER
        ),
        "activation_pca_dimension": (
            CFG.activation_pca_dimension
        ),
        "causal_point_model": (
            BEST_POINT_MODEL_NAME
        ),
        "causal_hazard_model": (
            BEST_HAZARD_MODEL_NAME
        ),
        "hazard_threshold": (
            HAZARD_DECISION_THRESHOLD
        ),
        "combined_hazard_threshold": (
            COMBINED_HAZARD_THRESHOLD
        ),
        "novelty_threshold": (
            NOVELTY_THRESHOLD
        ),
        "policy": (
            "RAW causal forecasting + "
            "separated transition-residual/"
            "activation-assisted novelty sensing"
        ),
    },
    "preflight": (
        preflight_results
    ),
    "activation_observability": {
        "layer_results": (
            layer_observability_results
        ),
        "selected_layer": (
            BEST_FEEDBACK_LAYER
        ),
    },
    "model_selection": {
        "point": (
            point_selection_results
        ),
        "hazard": (
            hazard_selection_results
        ),
    },
    "probability_and_policy_calibration": {
        "selected_hazard_threshold": (
            HAZARD_DECISION_THRESHOLD
        ),
        "selected_policy": (
            threshold_selection
        ),
        "combined_hazard_threshold": (
            COMBINED_HAZARD_THRESHOLD
        ),
        "combined_policy": (
            combined_threshold_selection
        ),
    },
    "untouched_validation": {
        "point": (
            validation_point_metrics
        ),
        "point_oracle": (
            validation_oracle_point_metrics
        ),
        "persistence": (
            validation_persistence_metrics
        ),
        "hazard": (
            validation_hazard_metrics
        ),
        "hazard_ap_gain": (
            validation_hazard_ap_gain
        ),
        "hazard_oracle": (
            validation_oracle_hazard_metrics
        ),
        "constant_brier": (
            validation_constant_brier
        ),
        "event_audit": (
            validation_event_audit
        ),
    },
    "horizon_specific_results": (
        horizon_results
    ),
    "temporal_order_ablation": {
        "point": {
            "ordered": (
                ordered_point_rho
            ),
            "reversed": (
                reversed_point_rho
            ),
            "shuffle_p95": (
                point_shuffle_p95
            ),
            "circular_p95": (
                point_circular_p95
            ),
            "advantage": (
                point_order_advantage
            ),
        },
        "hazard": {
            "ordered": (
                ordered_hazard_auc
            ),
            "reversed": (
                reversed_hazard_auc
            ),
            "shuffle_p95": (
                hazard_shuffle_p95
            ),
            "circular_p95": (
                hazard_circular_p95
            ),
            "advantage": (
                hazard_order_advantage
            ),
        },
    },
    "cross_regime_transfer": (
        transfer_results
    ),
    "novelty": {
        "auroc": novelty_auc,
        "auprc": novelty_ap,
        "validation_known_fpr": (
            validation_novelty_fpr
        ),
        "known_transfer_fpr": (
            known_transfer_novelty_fpr
        ),
        "unknown_tpr": (
            unknown_novelty_tpr
        ),
        "threshold": (
            NOVELTY_THRESHOLD
        ),
        "group_results": {
            group_name: {
                "mean_score": float(
                    np.mean(score)
                ),
                "tpr": float(
                    np.mean(
                        score
                        >= NOVELTY_THRESHOLD
                    )
                ),
            }
            for group_name, score in (
                unknown_group_scores.items()
            )
        },
    },
    "novelty_degradation_disentanglement": {
        "generator_phi": (
            generator_novelty_degradation_phi
        ),
        "detector_phi": (
            detector_degradation_phi
        ),
    },
    "channel_separation": (
        channel_separation_results
    ),
    "bootstrap_intervals": (
        bootstrap_intervals
    ),
    "scientific_gates": (
        scientific_gates
    ),
    "overall_statuses": (
        overall_statuses
    ),
    "diagnostic_messages": (
        diagnostic_messages
    ),
}

with open(
    CFG.output_file,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        to_builtin(
            telemetry
        ),
        file,
        ensure_ascii=False,
        indent=2,
    )


# =================================================================================================
# 29. CLEANUP
# =================================================================================================

try:
    del model
except Exception:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print(
    f"\n💾 Telemetry written to: "
    f"{CFG.output_file}"
)

print_header(
    "✅ M20.6.3.2 EXECUTION COMPLETED"
)

print(
    f"Final scientific status: "
    f"{'PASS' if overall_valid else 'FAIL'}"
)


🔬 MILESTONE 20.6.3.2 — STABLE-MECHANISM STRUCTURAL-ORACLE PREFLIGHT
Device                 : CUDA
Model                  : Qwen/Qwen2.5-0.5B
History length         : 8
Forecast horizons      : (1, 2, 3, 6)
Primary horizon        : 3
Latent threshold       : 0.72
Global seed            : 206322

🧪 PREFLIGHT A — STRUCTURAL FORECASTABILITY
H=1  | Point rho=0.9946 | Macro=0.9915 | R²=0.9901 | Hazard AUC=0.9765 | AP=0.7796 | Prev=0.0708 | PASS
H=2  | Point rho=0.9894 | Macro=0.9853 | R²=0.9807 | Hazard AUC=0.9684 | AP=0.8462 | Prev=0.1350 | PASS
H=3  | Point rho=0.9867 | Macro=0.9822 | R²=0.9758 | Hazard AUC=0.9690 | AP=0.8931 | Prev=0.1978 | PASS
H=6  | Point rho=0.9832 | Macro=0.9777 | R²=0.9699 | Hazard AUC=0.9717 | AP=0.9556 | Prev=0.3796 | PASS

✅ Primary benchmark is structurally forecastable. Activation extraction may proceed.

⏳ LOADING TRANSFORMERLENS MODEL


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
✅ Model loaded. Valid layers: [0, 3, 6, 9, 13, 16, 19, 23]

🔬 Extracting train/selection activations for layer selection...

🧪 TEST A — ACTIVATION-ONLY FEEDBACK OBSERVABILITY
Layer 0  | rho=0.8109 | R²=0.7923 | MAE=0.2278
Layer 3  | rho=0.8239 | R²=0.8048 | MAE=0.2194
Layer 6  | rho=0.9208 | R²=0.8729 | MAE=0.1697
Layer 9  | rho=0.9268 | R²=0.8779 | MAE=0.1652
Layer 13 | rho=0.9343 | R²=0.8878 | MAE=0.1595
Layer 16 | rho=0.9322 | R²=0.8857 | MAE=0.1618
Layer 19 | rho=0.9322 | R²=0.8859 | MAE=0.1596
Layer 23 | rho=0.9271 | R²=0.8787 | MAE=0.1642

🏆 Selected activation-only feedback layer: 13
  Extracting CALIBRATION_PROBABILITY...
  Extracting CALIBRATION_POLICY...
  Extracting VALIDATION...
  Extracting AR2_TRANSFER...
  Extracting DELAYED_TRANSFER...
  Extracting OSCILLATORY_TRANSFER...
  Extracting MEAN_REVERTING_TRANSFER...
  Extracting ABRUPT_UNKNOWN...
  Extracting MECHANISM_NOVEL...

🧪 TEST B — DISENTANGLED MODEL SE